# Pipeline Unificada Banco Central - SCR e Desenrola Brasil
Este notebook unifica a ingestão dos dados públicos do **SCR (Sistema de Informações de Crédito)** e do programa **Desenrola Brasil** para um mesmo schema no Databricks. As funções utilitárias foram componentizadas para garantir reutilização, padronização de metadados e total **idempotência** em tabelas Delta Lake distintas.

## Fonte de dados 

Como fonte de dados para esse trabalho, foram usados os dados abertos do Banco Central do Brasil, baixando direto do servidor disponibilizado no site https://dadosabertos.bcb.gov.br/ 

### SCR 

O Banco Central do Brasil divulga mensalmente informações agregadas das operações de crédito recebidas através do Sistema de Informações de Créditos – SCR.  Os relatórios são atualizados no último dia útil do mês, disponibilizando os dados após 30 dias do fechamento de cada período. Os arquivos .ZIP com os dados mensais da Versão 2 do SCR.Data podem ser baixados a partir da seguinte URL: https://www.bcb.gov.br/pda/desig/scrdata_{ANO}.zip

Foi criada uma função para baixar os últimos 5 anos de dados do SCR para compor a massa para análise histórica

### Desenrola

O Banco Central do Brasil (BCB) divulga mensalmente, por meio de arquivo CSV, dados agregados das operações de crédito informadas no Sistema de Informações de Créditos - SCR, vinculadas ao:

(¹) Programa Emergencial de Renegociação de Dívidas de Pessoas Físicas Inadimplentes - Desenrola Brasil, instituído pela Lei No 14.690, de 3 de outubro de 2023.
(²) Programa Desenrola Pequenos Negócios, instituído pela MP No 1.213, de 23 de abril de 2024.

### Licença dos Dados
Os conjuntos de dados utilizados neste projeto são disponibilizados pelo Banco Central do Brasil (BCB) em seu portal de dados abertos sob a Open Data Commons Open Database License (ODbL).
A ODbL é uma licença aberta destinada a bancos de dados, permitindo seu uso, compartilhamento e adaptação, observadas as condições estabelecidas pela licença, incluindo atribuição e compartilhamento nos termos aplicáveis.
Neste projeto, os dados são utilizados para fins acadêmicos e analíticos, mantendo-se a atribuição ao Banco Central do Brasil (BCB) como fonte dos dados.
Licença: Open Data Commons Open Database License (ODbL)
Fonte dos dados: Banco Central do Brasil — Dados Abertos
Detalhes da licença: [Open Data Commons Open Database License](https://opendefinition.org/licenses/odc-odbl/) (ODbL)

## Objetivo desse projeto

O projeto tem como objetivo analisar dados públicos do Sistema de Informações de Crédito (SCR) e do programa Desenrola Brasil para compreender a distribuição regional da carteira inadimplente, identificar as modalidades de crédito com maior participação na inadimplência e analisar o comportamento das renegociações de dívidas no país. 

### Perguntas a serem respondidas

Qual região apresentou o maior volume financeiro de carteira inadimplente ao longo de 2025?
R: A região Sudeste apresentou o maior volume financeiro de carteira inadimplente ao longo de 2025, encerrando o ano em aproximadamente R$ 120 bilhões, seguida pelas regiões Nordeste e Sul, ambas próximas de R$ 40 bilhões.

Qual modalidade de crédito representou a maior parcela da carteira vencida há mais de 90 dias em 2025?
R: A modalidade Empréstimos apresentou a maior participação na carteira vencida acima de 90 dias em 2025, representando aproximadamente 75,5% do total analisado, seguida por Financiamentos Rurais e Financiamentos.

Qual região concentrou o maior número de negociações do programa Desenrola Brasil ao longo de 2025?
R: A região Sudeste apresentou o maior número de negociações, atingindo um pico de aproximadamente 60 mil operações em outubro de 2025.

Qual conglomerado financeiro concentrou o maior número de negociações do programa Desenrola Brasil em 2025?
R: A Caixa Econômica Federal concentrou a maior participação no número de negociações em 2025, com 74,16% das operações analisadas, seguida por Itaú (17,84%) e Santander (2,66%).

## Discussão

### SCR: 

A partir dos dados, é possível observar que a modalidade Empréstimos concentra a maior parcela da carteira vencida há mais de 90 dias, representando aproximadamente 75,5% do total analisado. Considerando a carteira de empréstimos analisada, aproximadamente 92,66% encontra-se em dia e 7,34% corresponde à carteira inadimplente.

### Desenrola: 


As operações destinadas a Pessoas Jurídicas com faturamento de até R$ 4,8 milhões representam aproximadamente 72,3% do volume financeiro negociado no período analisado. 
É possível notar também que tivemos picos de negociações em 3 períodos específicos, tendo uma média de operações mensais abaixo das 5 miorda operações
porém em março tivemos um pico de aproximadamente 25 mil operações em agosto um pico similar, de ~25k operações e em outubro um pico de aproximadamente 60 mil operações


# Documentação Técnica

## Visão Geral da Arquitetura

Este notebook implementa um pipeline de dados unificado que ingere, transforma e modela informações públicas do Banco Central do Brasil (BCB) — especificamente os datasets do **SCR (Sistema de Informações de Crédito)** e do programa **Desenrola Brasil** — utilizando a **arquitetura medalhão** em três camadas (Bronze, Silver e Gold) sobre o Databricks Lakehouse.

O pipeline é executado de ponta a ponta por um bloco orquestrador central (Cell 30) que invoca sequencialmente cada estágio, garantindo que as dependências entre camadas sejam respeitadas: Bronze antes da Silver, Silver antes da Gold, e dimensões antes das tabelas fato.

---

## Arquitetura Medalhão

A arquitetura medalhão (medallion architecture) organiza os dados em camadas de maturidade crescente, onde cada camada adiciona estrutura, qualidade e valor analítico sobre a anterior:

| Camada | Função | Padrão de Escrita | Tabelas |
| --- | --- | --- | --- |
| **Bronze** | Ingestão raw dos dados no formato original da fonte, com metadados de auditoria injetados | `replaceWhere` por partição (SCR) e `overwrite` completo (Desenrola) | `brz_bcb_scr`, `brz_bcb_desenrola` |
| **Silver** | Limpeza, tipagem, padronização de nomes (UPPERCASE) e remoção de metadados de auditoria | `overwrite` com `overwriteSchema` | `slv_bcb_scr`, `slv_bcb_desenrola` |
| **Gold** | Modelo dimensional (Star Schema) com surrogate keys, dimensões compartilhadas e tabelas fato agregadas | Dimensões: `append` incremental via *left anti join* · Fatos: `overwrite` com agregação `GROUP BY` | 5 dimensões + 1 fato (SCR) · 1 dimensão + 1 fato (Desenrola) |

### Detalhamento das Camadas

**Bronze — Ingestão Raw**

A camada Bronze recebe os dados exatamente como publicados pelo BCB, sem transformações de conteúdo. Cada registro é enriquecido automaticamente com três metadados de auditoria:

* `nome_arquivo_origem` — caminho do arquivo-fonte processado (via `_metadata.file_path` do Spark)
* `data_processamento` — timestamp da carga (`current_timestamp`)
* `origem_carga` — identificador fixo da fonte (`BCB_PDA`)

O SCR é ingerido anualmente (arquivos ZIP por ano, últimos 5 anos) e particionado por `ano_carga`. O Desenrola é ingerido como um CSV único e consolidado, sobrescrevendo a tabela a cada carga.

**Silver — Transformação e Limpeza**

A camada Silver aplica tipagem consistente (`DECIMAL(18,2)` para colunas de saldo, `DATE` para datas), remove colunas de auditoria e indexadores não utilizados, padroniza nomes de colunas para UPPERCASE e realiza a conversão de formato decimal brasileiro (vírgula → ponto) via `regexp_replace`.

**Gold — Modelo Dimensional (Star Schema)**

A camada Gold implementa um modelo dimensional completo com tabelas dimensão e tabelas fato. As dimensões utilizam surrogate keys sequenciais e são atualizadas de forma idempotente via *left anti join* — apenas novos registros recebem IDs incrementais (`max(id) + row_number()`), preservando a estabilidade referencial das chaves já atribuídas. As tabelas fato agregam medidas (`SUM`) por combinação de chaves dimensionais, eliminando duplicação por design. Dimensões temporais (`dim_data_base`) e geográficas (`dim_uf`) são compartilhadas entre SCR e Desenrola.

---

## Garantia de Consistência dos Dados

A consistência dos dados baixados diretamente dos web services do BCB é garantida por um conjunto de mecanismos implementados em múltiplos níveis do pipeline:

### 1. Idempotência de Carga

* **SCR**: Cada ano é carregado como uma partição independente via `mode("overwrite")` + `option("replaceWhere", f"ano_carga = {ano}")`, permitindo que a reexecução de um ano específico substitua atomicamente apenas os dados daquela partição, sem afetar os demais anos já persistidos.
* **Desenrola**: O arquivo CSV é cumulativo/consolidado, então a tabela é inteiramente sobrescrita (`mode=overwrite` + `overwriteSchema=true`) a cada execução, garantindo que o estado final reflita sempre o snapshot mais recente publicado pelo BCB.

### 2. Integridade de Surrogate Keys

* A função `obter_max_id` calcula o próximo ID disponível excluindo a partição sendo substituída (no caso do SCR), evitando lacunas nos IDs em reexecuções.
* A função `adicionar_id_incremental` utiliza `row_number()` sobre uma janela global ordenada por `monotonically_increasing_id()`, garantindo IDs sequenciais não-nulos em todas as condições.
* Um verificador de integridade na camada Bronze detecta registros com `id` nulo (decorrentes de evolução de schema ou falhas anteriores) e dropa a tabela para forçar recriação completa com IDs válidos.

### 3. Idempotência de Dimensões (Gold)

* A função `criar_dimensao_idempotente` implementa o padrão *left anti join*: novos registros (chaves naturais não presentes na dimensão existente) recebem IDs incrementais a partir de `max(id) + 1`, enquanto registros já existentes mantêm seus IDs originais — garantindo estabilidade referencial para as tabelas fato.
* Validação de duplicação na chave natural ao final de cada carga dimensional: compara `count()` total vs `count(distinct)` da chave natural.

### 4. Integridade Referencial (Gold)

* As tabelas fato são validadas quanto à nulidade de todas as foreign keys após os joins com as dimensões, garantindo que nenhum registro fato fique órfão.
* Validação de duplicação na granularidade da fato: compara `count()` total vs `count(distinct)` da combinação de chaves dimensionais.

### 5. Resiliência de Download

* A função `download_arquivo` utiliza streaming (`requests.get(stream=True)`) com timeout de 60 segundos e tratamento de exceções, registrando avisos para arquivos indisponíveis (status HTTP não-200) sem interromper o pipeline.
* Falhas em um ano específico do SCR são isoladas por bloco `try/except` por iteração, permitindo que os demais anos sejam processados normalmente.

---

## Catálogo de Dados

O catálogo de dados completo, documentado na célula final deste notebook, descreve as **13 tabelas** persistidas no schema `mvp_pos_puc.default`, organizadas pelas três camadas da arquitetura medalhão:

| Camada | Tabela | Tipo | Registros |
| --- | --- | --- | --- |
| Bronze | `brz_bcb_scr` | Ingestão raw (particionada por `ano_carga`) | 14.194.430 |
| Bronze | `brz_bcb_desenrola` | Ingestão raw (overwrite completo) | 12.751 |
| Silver | `slv_bcb_scr` | Transformação e tipagem | 14.194.430 |
| Silver | `slv_bcb_desenrola` | Transformação e tipagem | 12.751 |
| Gold | `dim_data_base` | Dimensão temporal (compartilhada SCR + Desenrola) | 84 |
| Gold | `dim_uf` | Dimensão geográfica com região (compartilhada) | 27 |
| Gold | `dim_segmento` | Dimensão de segmento (SCR) | 8 |
| Gold | `dim_tipo_cliente` | Dimensão de cliente + porte (SCR) | variável |
| Gold | `dim_ocupacao` | Dimensão de CNAE/ocupação (SCR) | variável |
| Gold | `dim_modalidade` | Dimensão de modalidade + submodalidade (SCR) | variável |
| Gold | `dim_conglomerado` | Dimensão de conglomerado financeiro (Desenrola) | variável |
| Gold | `fct_taxa_de_endividamento` | Tabela fato agregada (SCR) | 9.193.430 |
| Gold | `fct_desenrola_negociacao` | Tabela fato agregada (Desenrola) | 12.751 |

Cada entrada do catálogo especifica descrição, granularidade, forma de carga, schema completo (coluna, tipo, nulabilidade e descrição), e transformações aplicadas — servindo como referência única para consumidores e governança.

---

## Agendamento (Scheduler)

O pipeline é automatizado via **Lakeflow Jobs** (Job ID: `381248367982685`) com as seguintes configurações:

| Parâmetro | Valor |
| --- | --- |
| Nome do Job | `pipeline_unificada_bcb` |
| Notebook vinculado | `/Users/lfandrade.si@gmail.com/Drafts/pipeline_unificada_bcb` |
| Expressão Cron | `14 0 13 27 * ?` (Quartz) |
| Frequência | Mensal — dia 27 de cada mês às 13:00:14 (horário de Brasília) |
| Fuso horário | `America/Sao_Paulo` |
| Execuções concorrentes | Máximo 1 (evita sobreposição de cargas) |
| Fila (Queue) | Habilitada — enfileira execuções disparadas durante outra execução |
| Otimização de performance | `PERFORMANCE_OPTIMIZED` |
| Notificação de falha | E-mail para `lfandrade.si@gmail.com` |
| Alerta para runs pulados | Suprimido (`no_alert_for_skipped_runs = true`) |

### Padrões de Agendamento

* **Frequência mensal no dia 27**: O SCR publica dados mensais com aproximadamente 30 dias de defasagem após o fechamento do período. O agendamento no dia 27 garante que os dados do mês anterior já estejam disponíveis no portal de dados abertos do BCB no momento da execução.
* **Execução única (max concurrent = 1)**: Previne que duas execuções do pipeline concorram pelos mesmos recursos de compute e evite condições de corrida nas tabelas Delta.
* **Queue habilitada**: Se uma execução for disparada enquanto outra ainda está em andamento (ex: execução manual acionada durante a janela agendada), a nova execução é enfileirada e processada sequencialmente.
* **Notificação por e-mail em falha**: O proprietário é notificado automaticamente caso a execução falhe, permitindo intervenção rápida.

---

## Visualizações Analíticas (Camada Gold)

Ao final do pipeline, **10 gráficos analíticos** são gerados com `matplotlib` a partir de consultas SQL sobre as tabelas fato e dimensão da camada Gold. As visualizações são agrupadas em dois eixos temáticos:

### Eixo SCR — Endividamento e Inadimplência

| # | Gráfico | Tipo | Período | Descrição |
| --- | --- | --- | --- | --- |
| 1 | Endividamento de Longo Prazo por Modalidade | Pizza | 2025 | Distribuição do saldo vencido acima de 90 dias por modalidade de crédito, com agrupamento de categorias < 2% em "Outros" |
| 2 | Endividamento por Modalidade e Tipo de Cliente | Pizza dupla (PF / PJ) | 2025 | Comparação lado a lado do endividamento de longo prazo entre Pessoa Física e Pessoa Jurídica, por modalidade |
| 3 | Taxa de Endividamento ao Longo do Ano | Linha (eixo duplo) | 2025 | Evolução mensal das taxas de endividamento (vencida/ativa) e inadimplência (inadimplência/ativa), com volume da carteira ativa em eixo secundário |
| 4 | Inadimplência por Região | Linha | 2025 | Evolução mensal da inadimplência (R$) por região do Brasil |
| 5 | Empréstimos: Inadimplência vs Carteira em Dia | Pizza | 2025 | Proporção entre carteira adimplente e inadimplente na modalidade de Empréstimos |
| 6 | Inadimplência por Região (Histórico) | Barras agrupadas | 2023–2025 | Comparativo mensal da carteira de inadimplência por região em perspectiva histórica trianual |

### Eixo Desenrola Brasil — Negociações e Renegociação

| # | Gráfico | Tipo | Período | Descrição |
| --- | --- | --- | --- | --- |
| 7 | Conglomerados que Mais Negociaram | Pizza | 2025 | Top 10 conglomerados financeiros por número de operações no Desenrola, com agrupamento dos demais em "Outros" |
| 8 | Negociações por Região | Linha | 2025 | Evolução mensal do número de negociações do Desenrola por região |
| 9 | Volume de Operações por Tipo | Pizza | 2025 | Distribuição do volume financeiro (R$) das operações do Desenrola por tipo de programa (PF até 2 salários, PF até 20 mil, PJ até 4,8 mi) |
| 10 | Negociações por Região (Histórico) | Barras agrupadas | 2023–2025 | Comparativo mensal do número de negociações por região em perspectiva histórica trianual |

### Padrões Técnicos das Visualizações

* Todas as consultas partem das tabelas fato da camada Gold, realizando joins com as dimensões (`dim_data_base`, `dim_uf`, `dim_modalidade`, `dim_tipo_cliente`, `dim_conglomerado`) para obter atributos descritivos.
* Os dados são convertidos para pandas (`toPandas()`) apenas no momento da plotagem, após a agregação no Spark, seguindo o padrão de processar pesado no cluster e plotar localmente.
* Categorias com representação inferior a 2% são automaticamente agrupadas em "Outros" para legibilidade dos gráficos de pizza.
* Formatação de eixos em escala brasileira (R$ em bilhões, percentuais com separador adequado) via `matplotlib.ticker.FuncFormatter`.



# Apendice Databriks

### Catalog 
![](/Workspace/Users/lfandrade.si@gmail.com/Drafts/catalog.png)

### Pipeline 
![](/Workspace/Users/lfandrade.si@gmail.com/Drafts/pipeline.png)

### Scheduler 
![](/Workspace/Users/lfandrade.si@gmail.com/Drafts/schedulers.png)




In [0]:
import os
import requests
import zipfile
import shutil
import time
from datetime import datetime
import pyspark.sql.functions as F

# CONFIGURAÇÃO DO SCHEMA UNIFICADO DE DESTINO
SCHEMA_DESTINO = "mvp_pos_puc.default"
TABELA_SCR = f"{SCHEMA_DESTINO}.brz_bcb_scr"
TABELA_DESENROLA = f"{SCHEMA_DESTINO}.brz_bcb_desenrola"

# URL DO DESENROLA
URL_DESENROLA = "https://www.bcb.gov.br/pda/desig/desenrola/dados_desenrola.csv"

# DIRETÓRIO RAIZ TEMPORÁRIO NO WORKSPACE
DIRETORIO_RAIZ_TMP = "/Workspace/tmp/bcb_pipeline_unificada"

# CRIAÇÃO E LEITURA DO PARÂMETRO PARA O SCR (WIDGET DATABRICKS)
QUANTIDADE_ANOS_SCR = 5
print(f"[Config] Quantidade de anos configurada para o SCR: {QUANTIDADE_ANOS_SCR}")
print(f"[Config] Tabela SCR: {TABELA_SCR}")
print(f"[Config] Tabela Desenrola: {TABELA_DESENROLA}")

### Funções Utilitárias Reutilizáveis (Core)

In [0]:
def obter_ultimos_anos(quantidade):
    """Calcula uma lista contendo o ano atual e os anos retroativos desejados."""
    ano_atual = datetime.now().year
    return [ano_atual - i for i in range(quantidade)]

def download_arquivo(url, caminho_destino):
    """Realiza o download de arquivos (CSV ou ZIP) de forma eficiente via stream."""
    try:
        resposta = requests.get(url, stream=True, timeout=60)
        if resposta.status_code == 200:
            with open(caminho_destino, 'wb') as f:
                for chunk in resposta.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"[Download] Sucesso: {url} -> {caminho_destino}")
            return True
        else:
            print(f"[Download] Aviso: Arquivo indisponível ou erro (Status: {resposta.status_code}) para a URL: {url}")
            return False
    except Exception as e:
        print(f"[Download] Erro ao tentar baixar {url}: {e}")
        return False

def ler_csv_bcb(diretorio_origem):
    """Lê CSVs do BCB via Spark para arquivos grandes (extraídos de ZIP).
    
    No compute serverless, arquivos criados via extração de ZIP são visíveis
    ao Spark através do protocolo file:. Para arquivos baixados diretamente
    via Python, use ler_csv_bcb_pandas."""
    caminho_spark_local = f"file:{diretorio_origem}"
    df = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("sep", ";")  # Padrão de delimitador de dados abertos do BCB
          .load(caminho_spark_local))
    
    # Injeção padronizada de metadados para governança e auditoria
    return df.withColumns({
        "nome_arquivo_origem": F.col("_metadata.file_path"),
        "data_processamento": F.current_timestamp(),
        "origem_carga": F.lit("BCB_PDA")
    })

def ler_csv_bcb_pandas(diretorio_origem):
    """Lê CSVs do BCB via pandas para arquivos pequenos, contornando a incompatibilidade
    do protocolo file: no Spark Connect serverless, onde arquivos recém-baixados
    via Python não são visíveis para o Spark (SharedUCWorkspaceLocalFileSystem).
    
    Usa pandas para ler diretamente do filesystem local e converte para Spark DataFrame."""
    import pandas as pd
    import glob
    
    arquivos = sorted(glob.glob(os.path.join(diretorio_origem, "*.csv")))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em {diretorio_origem}")
    
    dfs = []
    for arquivo in arquivos:
        df_pd = pd.read_csv(arquivo, sep=";", header=0)
        df_pd["nome_arquivo_origem"] = arquivo
        dfs.append(df_pd)
    
    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined["data_processamento"] = datetime.now()
    df_combined["origem_carga"] = "BCB_PDA"
    
    return spark.createDataFrame(df_combined)

def obter_max_id(tabela, coluna_id="id", coluna_particao=None, valor_particao=None):
    """Retorna o valor máximo atual da coluna de ID na tabela de destino.
    
    Retorna 0 se a tabela não existir ainda (primeira carga), permitindo
    que o auto-incremento comece em 1.
    
    Se coluna_particao e valor_particao forem fornecidos, exclui a partição
    sendo substituída (replaceWhere), evitando lacunas nos IDs em reexecuções."""
    try:
        df = spark.table(tabela)
        if coluna_particao is not None and valor_particao is not None:
            df = df.filter(F.col(coluna_particao) != valor_particao)
        max_id = df.agg(F.max(F.col(coluna_id))).collect()[0][0]
        return int(max_id) if max_id is not None else 0
    except Exception:
        return 0

def adicionar_id_incremental(df, max_id_atual, coluna_id="id"):
    """Adiciona uma coluna de ID auto-incremental sequencial ao DataFrame.
    
    Os IDs começam em max_id_atual + 1, garantindo continuidade entre cargas
    e unicidade para eliminação de duplicações. Utiliza row_number() sobre
    uma janela global ordenada por monotonically_increasing_id() para gerar
    IDs sequenciais (1, 2, 3, ...) não-nulos em todas as condições."""
    from pyspark.sql.window import Window
    w = Window.orderBy(F.monotonically_increasing_id())
    return df.withColumn(coluna_id, F.row_number().over(w) + F.lit(max_id_atual))

### Pipelines Específicas de Carga e Ingestão da Camada Bronze

In [0]:
def executar_pipeline_scr(quantidade_anos, tabela_destino):
    """Processa os arquivos anuais e compactados do SCR de forma idempotente por partição."""
    anos_para_processar = obter_ultimos_anos(quantidade_anos)
    print(f"\n=== INICIANDO PIPELINE SCR (Anos: {anos_para_processar}) ===")
    
    # VERIFICAÇÃO DE INTEGRIDADE: Se a tabela existe mas possui IDs nulos (decorrentes
    # de evolução de schema via mergeSchema ou falhas em cargas anteriores), dropa a
    # tabela para forçar recriação completa com IDs válidos em todas as partições
    try:
        null_count = spark.table(tabela_destino).filter(F.col("id").isNull()).count()
        if null_count > 0:
            print(f"[SCR] Aviso: {null_count} registros com id nulo encontrados. Dropando tabela para recriação completa.")
            spark.sql(f"DROP TABLE IF EXISTS {tabela_destino}")
    except Exception:
        pass  # Tabela ou coluna 'id' inexistente — primeira carga, prossegue normalmente
    
    for ano in anos_para_processar:
        url = f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip"
        diretorio_ano = os.path.join(DIRETORIO_RAIZ_TMP, f"scr_{ano}")
        caminho_zip = os.path.join(diretorio_ano, f"scrdata_{ano}.zip")
        diretorio_extraido = os.path.join(diretorio_ano, "extraido")
        
        try:
            os.makedirs(diretorio_extraido, exist_ok=True)
            
            if download_arquivo(url, caminho_zip):
                with zipfile.ZipFile(caminho_zip, 'r') as z:
                    z.extractall(diretorio_extraido)
                
                # Reutiliza o leitor padrão e adiciona a coluna de partição
                df_scr = ler_csv_bcb(diretorio_extraido).withColumns({"ano_carga": F.lit(ano)})
                
                # AUTO-INCREMENTO: Obtém o max ID atual excluindo a partição do ano corrente,
                # evitando lacunas nos IDs quando a partição é substituída via replaceWhere
                max_id_atual = obter_max_id(tabela_destino, coluna_particao="ano_carga", valor_particao=ano)
                df_scr = adicionar_id_incremental(df_scr, max_id_atual)
                
                # IDEMPOTÊNCIA SCR: Substituição focada e atômica da partição via replaceWhere
                (df_scr.write
                       .format("delta")
                       .mode("overwrite")
                       .option("replaceWhere", f"ano_carga = {ano}")
                       .option("mergeSchema", "true")
                       .saveAsTable(tabela_destino))
                
                print(f"[SCR] Sucesso: Dados do ano {ano} persistidos com idempotência.")
        except Exception as e:
            print(f"[SCR] Erro ao processar o ano {ano}: {e}")
        finally:
            if os.path.exists(diretorio_ano):
                shutil.rmtree(diretorio_ano)

def executar_pipeline_desenrola(url_origem, tabela_destino):
    """Processa o arquivo consolidado e único do Desenrola Brasil substituindo a tabela como um todo."""
    print(f"\n=== INICIANDO PIPELINE DESENROLA BRASIL ===")
    diretorio_desenrola = os.path.join(DIRETORIO_RAIZ_TMP, "desenrola")
    caminho_csv = os.path.join(diretorio_desenrola, "dados_desenrola.csv")
    
    try:
        os.makedirs(diretorio_desenrola, exist_ok=True)
        
        if download_arquivo(url_origem, caminho_csv):
            # Usa pandas para contornar a incompatibilidade do protocolo file: no serverless
            df_desenrola = ler_csv_bcb_pandas(diretorio_desenrola)
            
            # AUTO-INCREMENTO: Tabela é totalmente sobrescrita (mode=overwrite),
            # IDs recomeçam em 1 a cada carga para manter sequência limpa
            max_id_atual = 0
            df_desenrola = adicionar_id_incremental(df_desenrola, max_id_atual)
            
            # IDEMPOTÊNCIA DESENROLA: Arquivo cumulativo/consolidadado pede overwrite completo da tabela
            (df_desenrola.write
                         .format("delta")
                         .mode("overwrite")
                         .option("overwriteSchema", "true")
                         .saveAsTable(tabela_destino))
            
            print(f"[Desenrola] Sucesso: Tabela atualizada atomicamente de forma idempotente.")
    except Exception as e:
        print(f"[Desenrola] Erro ao processar dados: {e}")
    finally:
        if os.path.exists(diretorio_desenrola):
            shutil.rmtree(diretorio_desenrola)

In [0]:
# === QUALIDADE DE DADOS: TRATAMENTO DE NULOS E DETECCAO DE OUTLIERS ===
# Executado entre as camadas Bronze e Silver para garantir integridade dos dados.
# - Colunas numericas: nulos -> 0
# - Colunas string/texto: nulos -> "N/A"
# - Outliers detectados via metodo IQR e listados como informacao de qualidade

from pyspark.sql.types import (IntegerType, LongType, DoubleType, DecimalType,
                                FloatType, StringType)

# Colunas de auditoria/metadados que nao recebem tratamento de nulos
COLUNAS_METADADOS = {
    "nome_arquivo_origem", "data_processamento", "origem_carga",
    "id", "ano_carga"
}

def tratar_nulos_e_outliers(nome_tabela):
    """Trata valores nulos e detecta outliers em uma tabela Bronze.

    Regras de tratamento:
    - Colunas numericas: nulos preenchidos com 0
    - Colunas string/texto: nulos preenchidos com 'N/A'
    - Colunas de data/timestamp e metadados de auditoria: nao alteradas

    Deteccao de outliers:
    - Metodo IQR (Interquartile Range): Q1 - 1.5*IQR, Q3 + 1.5*IQR
    - Aplicado apenas a colunas numericas (excluindo id e metadados)
    - Resultados exibidos como informacao de qualidade dos dados
    """
    print(f"\n{'='*70}")
    print(f"[Qualidade] Tratamento de Nulos e Deteccao de Outliers: {nome_tabela}")
    print(f"{'='*70}")

    df = spark.table(nome_tabela)
    schema = df.schema

    # Classifica colunas por tipo, excluindo metadados
    colunas_numericas = []
    colunas_string = []
    colunas_ignoradas = []

    for campo in schema.fields:
        nome = campo.name
        tipo = campo.dataType

        if nome.lower() in COLUNAS_METADADOS:
            colunas_ignoradas.append(nome)
        elif isinstance(tipo, (IntegerType, LongType, DoubleType, FloatType, DecimalType)):
            colunas_numericas.append(nome)
        elif isinstance(tipo, StringType):
            colunas_string.append(nome)
        else:
            colunas_ignoradas.append(nome)

    print(f"  Colunas numericas  ({len(colunas_numericas)}): {colunas_numericas}")
    print(f"  Colunas string     ({len(colunas_string)}): {colunas_string}")
    print(f"  Colunas ignoradas  ({len(colunas_ignoradas)}): {colunas_ignoradas}")

    # --- 1. RELATORIO DE NULOS (unica passada via SQL) ---
    cols_check = colunas_numericas + colunas_string
    if cols_check:
        null_exprs = [f"SUM(CASE WHEN `{c}` IS NULL THEN 1 ELSE 0 END) AS `{c}__nulls`" for c in cols_check]
        null_exprs.append("COUNT(*) AS __total")
        query_nulls = f"SELECT {', '.join(null_exprs)} FROM {nome_tabela}"
        null_result = spark.sql(query_nulls).collect()[0].asDict()
        total_registros = null_result["__total"]

        nulos_antes = {}
        for c in cols_check:
            cnt = null_result.get(f"{c}__nulls", 0)
            if cnt > 0:
                nulos_antes[c] = cnt

        if nulos_antes:
            print(f"\n  Nulos encontrados antes do tratamento:")
            for c, cnt in sorted(nulos_antes.items(), key=lambda x: -x[1]):
                print(f"    {c}: {cnt:,} nulos")
        else:
            print(f"\n  Nenhum valor nulo encontrado nas colunas trataveis.")
    else:
        total_registros = df.count()
        print(f"\n  Nenhuma coluna tratavel encontrada.")

    # --- 2. APLICA TRATAMENTO DE NULOS ---
    # Usa withColumns (nao withColumn em loop) para compatibilidade com Spark Connect
    fill_exprs = {}
    for col in colunas_numericas:
        fill_exprs[col] = F.when(F.col(col).isNull(), F.lit(0)).otherwise(F.col(col))
    for col in colunas_string:
        fill_exprs[col] = F.when(F.col(col).isNull(), F.lit("N/A")).otherwise(F.col(col))

    if fill_exprs:
        df_tratado = df.withColumns(fill_exprs)
    else:
        df_tratado = df

    (df_tratado.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela))

    print(f"\n  [OK] Tabela {nome_tabela} tratada e sobrescrita com sucesso.")

    # --- 3. DETECCAO DE OUTLIERS (METODO IQR) ---
    if not colunas_numericas:
        print(f"\n  Nenhuma coluna numerica para analise de outliers.")
        return

    print(f"\n  Analise de Outliers (Metodo IQR) -- {total_registros:,} registros")
    print(f"  {'Coluna':<40} {'Q1':>15} {'Q3':>15} {'IQR':>15} {'Lim. Inferior':>15} {'Lim. Superior':>15} {'Outliers':>10} {'%':>7}")
    print(f"  {'-'*40} {'-'*15} {'-'*15} {'-'*15} {'-'*15} {'-'*15} {'-'*10} {'-'*7}")

    # Calcula Q1 e Q3 em uma unica passada via SQL percentile_approx
    select_exprs = []
    for col in colunas_numericas:
        select_exprs.append(f"percentile_approx(`{col}`, 0.25) AS `{col}__q1`")
        select_exprs.append(f"percentile_approx(`{col}`, 0.75) AS `{col}__q3`")

    query_percentis = f"SELECT {', '.join(select_exprs)} FROM {nome_tabela}"
    percentis = spark.sql(query_percentis).collect()[0].asDict()

    # Constroi query de contagem de outliers em uma unica passada
    count_exprs = []
    bounds = {}
    for col in colunas_numericas:
        q1 = percentis.get(f"{col}__q1")
        q3 = percentis.get(f"{col}__q3")
        if q1 is None or q3 is None:
            continue
        iqr = float(q3) - float(q1)
        lim_inf = float(q1) - 1.5 * iqr
        lim_sup = float(q3) + 1.5 * iqr
        bounds[col] = {"q1": q1, "q3": q3, "iqr": iqr, "lim_inf": lim_inf, "lim_sup": lim_sup}
        count_exprs.append(
            f"SUM(CASE WHEN `{col}` < {lim_inf} OR `{col}` > {lim_sup} THEN 1 ELSE 0 END) AS `{col}__outliers`"
        )

    if count_exprs:
        query_count = f"SELECT {', '.join(count_exprs)} FROM {nome_tabela}"
        outlier_counts = spark.sql(query_count).collect()[0].asDict()
    else:
        outlier_counts = {}

    for col in colunas_numericas:
        if col not in bounds:
            continue
        b = bounds[col]
        cnt_out = outlier_counts.get(f"{col}__outliers", 0)
        pct = (cnt_out / total_registros * 100) if total_registros > 0 else 0
        print(f"  {col:<40} {b['q1']:>15,.2f} {b['q3']:>15,.2f} {b['iqr']:>15,.2f} {b['lim_inf']:>15,.2f} {b['lim_sup']:>15,.2f} {cnt_out:>10,} {pct:>6.2f}%")

    print(f"\n  [OK] Analise de outliers concluida para {len(bounds)} colunas numericas.")


# --- EXECUCAO PARA AMBAS AS TABELAS BRONZE ---
TABELAS_BRONZE_QUALIDADE = [
    f"{SCHEMA_DESTINO}.brz_bcb_scr",
    f"{SCHEMA_DESTINO}.brz_bcb_desenrola"
]

for tabela in TABELAS_BRONZE_QUALIDADE:
    tratar_nulos_e_outliers(tabela)

print(f"\n{'='*70}")
print(f"[Qualidade] Tratamento de nulos e analise de outliers concluidos!")
print(f"{'='*70}")

### Camada Silver — Transformação da Tabela Desenrola

Cria a tabela `slv_bcb_desenrola` a partir da camada bronze `brz_bcb_desenrola`, removendo as colunas de metadados de auditoria (`nome_arquivo_origem`, `data_processamento`, `origem_carga`) e ajustando o formato da coluna `DATA_BASE` de `YYYYMM` para `DD/MM/YYYY` (com acréscimo do dia `01`).

In [0]:
# Tabelas de origem e destino
TABELA_BRONZE_DESENROLA = f"{SCHEMA_DESTINO}.brz_bcb_desenrola"
TABELA_SILVER_DESENROLA = f"{SCHEMA_DESTINO}.slv_bcb_desenrola"

def criar_silver_desenrola(tabela_bronze, tabela_silver):
    """Cria a tabela silver do Desenrola a partir da bronze.
    
    - Remove as colunas de metadados: nome_arquivo_origem, data_processamento, origem_carga
    - Transforma DATA_BASE de YYYYMM (long) para date, assumindo o dia 01 do mês
    - Converte TIPO_DESENROLA de inteiro para string descritiva:
        1 -> PF_ATE_2_SALARIOS, 2 -> PF_ATE_20_MIL, 3 -> PJ_FATURAMENTO_ATE_4_8_MM
    - Renomeia a coluna UNIDADE_FEDERACAO para UF
    """
    print(f"\n=== CRIANDO TABELA SILVER DESENROLA ===")
    print(f"[Silver] Origem:  {tabela_bronze}")
    print(f"[Silver] Destino: {tabela_silver}")
    
    try:
        df_silver = (spark.table(tabela_bronze)
            .drop("nome_arquivo_origem", "data_processamento", "origem_carga")
            # DATA_BASE vem como LONG no formato YYYYMM (ex: 202312)
            # Converte para date, assumindo o dia 01 do mês
            .withColumn("DATA_BASE",
                F.to_date(F.col("DATA_BASE").cast("string"), "yyyyMM"))
            # TIPO_DESENROLA: inteiro -> string descritiva
            .withColumn("TIPO_DESENROLA",
                F.when(F.col("TIPO_DESENROLA") == 1, F.lit("PF_ATE_2_SALARIOS"))
                 .when(F.col("TIPO_DESENROLA") == 2, F.lit("PF_ATE_20_MIL"))
                 .when(F.col("TIPO_DESENROLA") == 3, F.lit("PJ_FATURAMENTO_ATE_4_8_MM"))
                 .otherwise(F.col("TIPO_DESENROLA").cast("string")))
            # Renomeia UNIDADE_FEDERACAO -> UF
            .withColumnRenamed("UNIDADE_FEDERACAO", "UF")
        )
        
        # Padroniza todos os nomes de colunas para uppercase
        df_silver = df_silver.toDF(*[c.upper() for c in df_silver.columns])
        
        (df_silver.write
                 .format("delta")
                 .mode("overwrite")
                 .option("overwriteSchema", "true")
                 .saveAsTable(tabela_silver))
        
        print(f"[Silver] Sucesso: Tabela {tabela_silver} criada/atualizada.")
        
    except Exception as e:
        print(f"[Silver] Erro ao criar tabela silver: {e}")

# Executa a transformação
criar_silver_desenrola(TABELA_BRONZE_DESENROLA, TABELA_SILVER_DESENROLA)

### Camada Silver — Transformação da Tabela SCR

Cria a tabela `slv_bcb_scr` a partir da camada bronze `brz_bcb_scr`, removendo as colunas `indexador`, `nome_arquivo_origem`, `data_processamento`, `origem_carga` e `ano_carga`, e ajustando o formato da coluna `data_base` (tipo `date`) para `DD/MM/YYYY` (string).

In [0]:
# Tabelas de origem e destino
TABELA_BRONZE_SCR = f"{SCHEMA_DESTINO}.brz_bcb_scr"
TABELA_SILVER_SCR = f"{SCHEMA_DESTINO}.slv_bcb_scr"

# Colunas de saldo que devem ser convertidas de string para decimal(18,2)
COLUNAS_SALDO_DECIMAL = [
    "numero_de_operacoes",
    "a_vencer_ate_90_dias",
    "a_vencer_de_91_ate_360_dias",
    "a_vencer_de_361_ate_1080_dias",
    "a_vencer_de_1081_ate_1800_dias",
    "a_vencer_de_1801_ate_5400_dias",
    "a_vencer_acima_de_5400_dias",
    "carteira_a_vencer",
    "vencido_de_15_ate_90_dias",
    "vencido_acima_de_90_dias",
    "carteira_vencida",
    "carteira_ativa",
    "carteira_inadimplencia",
    "ativo_problematico",
]

def criar_silver_scr(tabela_bronze, tabela_silver):
    """Cria a tabela silver do SCR a partir da bronze.
    
    - Remove as colunas: indexador, nome_arquivo_origem, data_processamento, origem_carga, ano_carga
    - Mantém data_base como tipo date
    - Converte colunas de saldo de string para decimal(18,2), tratando separador
      decimal no formato brasileiro (vírgula -> ponto)
    """
    print(f"\n=== CRIANDO TABELA SILVER SCR ===")
    print(f"[Silver] Origem:  {tabela_bronze}")
    print(f"[Silver] Destino: {tabela_silver}")
    
    try:
        df_silver = (spark.table(tabela_bronze)
            .drop("indexador", "nome_arquivo_origem", "data_processamento", "origem_carga", "ano_carga")
        )
        
        # Conversão de colunas de saldo: string -> decimal(18,2)
        # Substitui vírgula decimal (formato brasileiro) por ponto antes do cast
        conversoes_saldo = {
            col: F.regexp_replace(F.col(col), ",", ".").cast("decimal(18,2)")
            for col in COLUNAS_SALDO_DECIMAL
        }
        df_silver = df_silver.withColumns(conversoes_saldo)
        
        # Padroniza todos os nomes de colunas para uppercase
        df_silver = df_silver.toDF(*[c.upper() for c in df_silver.columns])
        
        (df_silver.write
                 .format("delta")
                 .mode("overwrite")
                 .option("overwriteSchema", "true")
                 .saveAsTable(tabela_silver))
        
        print(f"[Silver] Sucesso: Tabela {tabela_silver} criada/atualizada.")
        
    except Exception as e:
        print(f"[Silver] Erro ao criar tabela silver: {e}")

# Executa a transformação
criar_silver_scr(TABELA_BRONZE_SCR, TABELA_SILVER_SCR)

   
### Camada Gold — Tabelas Fato e Dimensão (SCR)

Cria as tabelas dimensionais (`dim_data_base`, `dim_uf`, `dim_segmento`, `dim_tipo_cliente`, `dim_ocupacao`) e a tabela fato (`fct_taxa_de_endividamento`) a partir da camada silver `slv_bcb_scr`.

- **Dimensões**: surrogate keys sequenciais, idempotentes via *left anti join* — novos registros recebem `max(id) + row_number()`, preservando IDs existentes para estabilidade referencial.
- **Fato**: agregação (`SUM`) das medidas por combinação de chaves dimensionais (`GROUP BY`), eliminando duplicação por design.
- **Validações**: verificação de duplicidade nas chaves naturais das dimensões e na granularidade da fato, além de integridade referencial (FKs não nulas).

In [0]:
# Configuração das tabelas da camada Gold
TABELA_SILVER_SCR = f"{SCHEMA_DESTINO}.slv_bcb_scr"
DIM_DATA_BASE = f"{SCHEMA_DESTINO}.dim_data_base"
DIM_UF = f"{SCHEMA_DESTINO}.dim_uf"
DIM_SEGMENTO = f"{SCHEMA_DESTINO}.dim_segmento"
DIM_TIPO_CLIENTE = f"{SCHEMA_DESTINO}.dim_tipo_cliente"
DIM_OCUPACAO = f"{SCHEMA_DESTINO}.dim_ocupacao"
DIM_MODALIDADE = f"{SCHEMA_DESTINO}.dim_modalidade"
TABELA_FCT_ENDIVIDAMENTO = f"{SCHEMA_DESTINO}.fct_taxa_de_endividamento"

from pyspark.sql.window import Window

# Mapeamento UF -> Região do Brasil
UF_REGIAO = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

def enriquecer_dim_uf_com_regiao(df_uf):
    """Adiciona a coluna 'regiao' ao DataFrame de UFs via broadcast de mapeamento."""
    df_regiao = spark.createDataFrame(
        [(uf, regiao) for uf, regiao in UF_REGIAO.items()],
        ["uf", "regiao"]
    )
    return df_uf.join(F.broadcast(df_regiao), on="uf", how="left")

def garantir_coluna_regiao_dim_uf():
    """Se dim_uf já existe sem a coluna 'regiao', dropa para recriar com o novo schema."""
    if tabela_existe(DIM_UF):
        cols_existing = spark.table(DIM_UF).columns
        if "regiao" not in cols_existing:
            spark.sql(f"DROP TABLE IF EXISTS {DIM_UF}")
            print(f"[Gold] Tabela {DIM_UF} droppada para adicionar coluna 'regiao'.")

def tabela_existe(nome_tabela):
    """Verifica se uma tabela existe (compatível com Spark Connect via ação)."""
    try:
        spark.table(nome_tabela).limit(1).collect()
        return True
    except Exception:
        return False

def criar_dimensao_idempotente(nome_tabela, id_col, df_distinct, colunas_natural_key):
    """Cria ou atualiza uma dimensão de forma idempotente.
    
    - Primeira carga: cria a tabela com IDs sequenciais a partir de 1.
    - Cargas subsequentes: adiciona apenas novos registros via left anti join,
      preservando IDs existentes (estabilidade referencial para a fato).
    - Valida duplicação na chave natural ao final.
    """
    print(f"\n--- Dimensão: {nome_tabela} ---")
    
    if not tabela_existe(nome_tabela):
        # Primeira carga: atribui IDs sequenciais
        w = Window.orderBy(F.monotonically_increasing_id())
        df_dim = df_distinct.withColumn(id_col, F.row_number().over(w))
        (df_dim.write.format("delta").mode("overwrite").saveAsTable(nome_tabela))
        total = df_dim.count()
        print(f"[Gold] Criada com {total} registros (IDs 1 a {total}).")
    else:
        # Carga incremental: apenas novos registros (left anti join)
        df_existing = spark.table(nome_tabela)
        existing_keys = df_existing.select(*colunas_natural_key)
        df_new = df_distinct.join(existing_keys, on=colunas_natural_key, how="left_anti")
        count_new = df_new.count()
        
        if count_new > 0:
            max_id_result = df_existing.agg(F.max(F.col(id_col))).collect()[0][0]
            max_id = int(max_id_result) if max_id_result is not None else 0
            w = Window.orderBy(F.monotonically_increasing_id())
            df_new = df_new.withColumn(id_col, F.row_number().over(w) + F.lit(max_id))
            (df_new.write.format("delta").mode("append").saveAsTable(nome_tabela))
            print(f"[Gold] {count_new} novos registros (IDs {max_id+1} a {max_id+count_new}).")
        else:
            print(f"[Gold] Sem novos registros. Total mantido.")
    
    # Validação de duplicação na chave natural
    df_check = spark.table(nome_tabela)
    total = df_check.count()
    distinto = df_check.select(*colunas_natural_key).distinct().count()
    if total != distinto:
        print(f"[Gold] AVISO: {total - distinto} duplicatas na chave natural!")
    else:
        print(f"[Gold] OK: {total} registros únicos validados.")

def criar_dimensoes_scr():
    """Cria ou atualiza todas as 5 dimensões da camada Gold do SCR."""
    print("\n=== INICIANDO CRIAÇÃO DAS DIMENSÕES GOLD — SCR ===")
    df_silver = spark.table(TABELA_SILVER_SCR)
    
    # 1. dim_data_base: DATA_BASE -> data_base, ano, mes
    df_data_base = (df_silver
        .select(F.col("DATA_BASE").alias("data_base"))
        .distinct()
        .withColumns({
            "ano": F.year(F.col("data_base")),
            "mes": F.month(F.col("data_base")),
        }))
    criar_dimensao_idempotente(DIM_DATA_BASE, "id_data_base", df_data_base, ["data_base"])
    
    # 2. dim_uf: UF -> uf, regiao (com mapeamento UF -> região)
    garantir_coluna_regiao_dim_uf()
    df_uf = enriquecer_dim_uf_com_regiao(df_silver.select(F.col("UF").alias("uf")).distinct())
    criar_dimensao_idempotente(DIM_UF, "id_uf", df_uf, ["uf"])
    
    # 3. dim_segmento: SEGMENTO -> segmento
    criar_dimensao_idempotente(DIM_SEGMENTO, "id_segmento",
        df_silver.select(F.col("SEGMENTO").alias("segmento")).distinct(), ["segmento"])
    
    # 4. dim_tipo_cliente: (CLIENTE, PORTE) -> (cliente, porte)
    criar_dimensao_idempotente(DIM_TIPO_CLIENTE, "id_tipo_cliente",
        df_silver.select(
            F.col("CLIENTE").alias("cliente"),
            F.col("PORTE").alias("porte")
        ).distinct(), ["cliente", "porte"])
    
    # 5. dim_ocupacao: CNAE_OCUPACAO -> cnae_ocupacao
    criar_dimensao_idempotente(DIM_OCUPACAO, "id_ocupacao",
        df_silver.select(F.col("CNAE_OCUPACAO").alias("cnae_ocupacao")).distinct(),
        ["cnae_ocupacao"])
    
    # 6. dim_modalidade: (MODALIDADE, SUBMODALIDADE) -> (modalidade, submodalidade)
    criar_dimensao_idempotente(DIM_MODALIDADE, "id_modalidade",
        df_silver.select(
            F.col("MODALIDADE").alias("modalidade"),
            F.col("SUBMODALIDADE").alias("submodalidade")
        ).distinct(), ["modalidade", "submodalidade"])
    
    print("\n=== DIMENSÕES GOLD — SCR FINALIZADAS ===")

# Executa a criação das dimensões
criar_dimensoes_scr()

# Amostra das dimensões criadas
print("\n--- Amostra dim_data_base ---")
display(spark.table(DIM_DATA_BASE).limit(10))
print("\n--- Amostra dim_tipo_cliente ---")
display(spark.table(DIM_TIPO_CLIENTE).limit(10))

In [0]:
def criar_fato_endividamento(tabela_silver, tabela_fato):
    """Cria a tabela fato de endividamento a partir da silver SCR + dimensões.
    
    - Joins com todas as dimensões para obter surrogate keys
    - Agrega medidas por combinação de chaves dimensionais (GROUP BY)
    - A agregação elimina duplicações naturalmente
    - Valida integridade referencial (FKs não nulas) e duplicação ao final
    """
    print(f"\n=== CRIANDO TABELA FATO: {tabela_fato} ===")
    
    df_silver = spark.table(tabela_silver)
    
    # Joins com dimensões usando aliases para evitar ambiguidade de colunas
    df_gold = (df_silver.alias("s")
        .join(spark.table(DIM_DATA_BASE).alias("ddb"),
              F.col("s.DATA_BASE") == F.col("ddb.data_base"), "left")
        .join(spark.table(DIM_UF).alias("du"),
              F.col("s.UF") == F.col("du.uf"), "left")
        .join(spark.table(DIM_SEGMENTO).alias("ds"),
              F.col("s.SEGMENTO") == F.col("ds.segmento"), "left")
        .join(spark.table(DIM_TIPO_CLIENTE).alias("dtc"),
              (F.col("s.CLIENTE") == F.col("dtc.cliente")) &
              (F.col("s.PORTE") == F.col("dtc.porte")), "left")
        .join(spark.table(DIM_OCUPACAO).alias("doc"),
              F.col("s.CNAE_OCUPACAO") == F.col("doc.cnae_ocupacao"), "left")
        .join(spark.table(DIM_MODALIDADE).alias("dm"),
              (F.col("s.MODALIDADE") == F.col("dm.modalidade")) &
              (F.col("s.SUBMODALIDADE") == F.col("dm.submodalidade")), "left")
        .select(
            F.col("ddb.id_data_base"),
            F.col("du.id_uf"),
            F.col("ds.id_segmento"),
            F.col("dtc.id_tipo_cliente"),
            F.col("doc.id_ocupacao"),
            F.col("dm.id_modalidade"),
            F.col("s.NUMERO_DE_OPERACOES"),
            F.col("s.CARTEIRA_ATIVA"),
            F.col("s.CARTEIRA_A_VENCER"),
            F.col("s.CARTEIRA_VENCIDA"),
            F.col("s.CARTEIRA_INADIMPLENCIA"),
            F.col("s.ATIVO_PROBLEMATICO"),
            F.col("s.A_VENCER_ATE_90_DIAS"),
            F.col("s.A_VENCER_DE_91_ATE_360_DIAS"),
            F.col("s.A_VENCER_DE_361_ATE_1080_DIAS"),
            F.col("s.A_VENCER_DE_1081_ATE_1800_DIAS"),
            F.col("s.A_VENCER_DE_1801_ATE_5400_DIAS"),
            F.col("s.A_VENCER_ACIMA_DE_5400_DIAS"),
            F.col("s.VENCIDO_DE_15_ATE_90_DIAS"),
            F.col("s.VENCIDO_ACIMA_DE_90_DIAS"),
        )
    )
    
    # Validação de integridade referencial: nenhum FK deve ser nulo
    fk_nulos = df_gold.filter(
        F.col("id_data_base").isNull() | F.col("id_uf").isNull() |
        F.col("id_segmento").isNull() | F.col("id_tipo_cliente").isNull() |
        F.col("id_ocupacao").isNull() | F.col("id_modalidade").isNull()
    ).count()
    
    if fk_nulos > 0:
        print(f"[Gold] AVISO: {fk_nulos} registros com FK nula — verificar dimensões.")
    else:
        print(f"[Gold] OK: Integridade referencial validada — todas as FKs preenchidas.")
    
    # Agregação por chaves dimensionais (elimina duplicação por design)
    df_fct = (df_gold
        .groupBy("id_data_base", "id_uf", "id_segmento", "id_tipo_cliente", "id_ocupacao", "id_modalidade")
        .agg(
            F.sum("NUMERO_DE_OPERACOES").alias("numero_de_operacoes"),
            F.sum("CARTEIRA_ATIVA").alias("carteira_ativa"),
            F.sum("CARTEIRA_A_VENCER").alias("carteira_a_vencer"),
            F.sum("CARTEIRA_VENCIDA").alias("carteira_vencida"),
            F.sum("CARTEIRA_INADIMPLENCIA").alias("carteira_inadimplencia"),
            F.sum("ATIVO_PROBLEMATICO").alias("ativo_problematico"),
            F.sum("A_VENCER_ATE_90_DIAS").alias("a_vencer_ate_90_dias"),
            F.sum("A_VENCER_DE_91_ATE_360_DIAS").alias("a_vencer_de_91_ate_360_dias"),
            F.sum("A_VENCER_DE_361_ATE_1080_DIAS").alias("a_vencer_de_361_ate_1080_dias"),
            F.sum("A_VENCER_DE_1081_ATE_1800_DIAS").alias("a_vencer_de_1081_ate_1800_dias"),
            F.sum("A_VENCER_DE_1801_ATE_5400_DIAS").alias("a_vencer_de_1801_ate_5400_dias"),
            F.sum("A_VENCER_ACIMA_DE_5400_DIAS").alias("a_vencer_acima_de_5400_dias"),
            F.sum("VENCIDO_DE_15_ATE_90_DIAS").alias("vencido_de_15_ate_90_dias"),
            F.sum("VENCIDO_ACIMA_DE_90_DIAS").alias("vencido_acima_de_90_dias"),
        )
    )
    
    # Escrita idempotente: overwrite (a agregação é determinística a partir da silver)
    (df_fct.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(tabela_fato))
    
    # Validação de duplicação: a combinação de todas as FKs deve ser única
    total = spark.table(tabela_fato).count()
    distinto = (spark.table(tabela_fato)
        .select("id_data_base", "id_uf", "id_segmento", "id_tipo_cliente", "id_ocupacao", "id_modalidade")
        .distinct().count())
    
    if total != distinto:
        print(f"[Gold] AVISO: {total - distinto} duplicatas na tabela fato!")
    else:
        print(f"[Gold] OK: {total} registros únicos sem duplicação.")
    
    print(f"[Gold] Fato {tabela_fato} criada com {total} registros.")

# Executa a criação da fato
criar_fato_endividamento(TABELA_SILVER_SCR, TABELA_FCT_ENDIVIDAMENTO)

# Amostra da fato criada
print("\n--- Amostra fct_taxa_de_endividamento ---")
display(spark.table(TABELA_FCT_ENDIVIDAMENTO).limit(10))

   
### Camada Gold — Tabelas Fato e Dimensão (Desenrola)

Cria a dimensão específica `dim_conglomerado` e a tabela fato `fct_desenrola_negociacao` a partir da camada silver `slv_bcb_desenrola`.

- **Dimensões compartilhadas**: `dim_data_base` e `dim_uf` são atualizadas com novos valores do Desenrola (left anti join), preservando os IDs já atribuídos pelo SCR.
- **Dimensão específica**: `dim_conglomerado` (cod_conglomerado_financeiro + nome_conglomerado_financeiro) com surrogate keys idempotentes.
- **Fato**: agregação (`SUM`) das medidas por combinação de chaves dimensionais + `tipo_desenrola`, eliminando duplicação por design. Conversão de `VOLUME_OPERACOES` de string (vírgula decimal) para `decimal(18,2)`.
- **Validações**: integridade referencial (FKs não nulas), duplicação na granularidade da fato e duplicação nas chaves naturais da dimensão.

In [0]:
# Configuração das tabelas Gold — Desenrola
TABELA_SILVER_DESENROLA = f"{SCHEMA_DESTINO}.slv_bcb_desenrola"
DIM_CONGLOMERADO = f"{SCHEMA_DESTINO}.dim_conglomerado"
TABELA_FCT_DESENROLA = f"{SCHEMA_DESTINO}.fct_desenrola_negociacao"

def criar_dimensoes_desenrola():
    """Cria/atualiza as dimensões da camada Gold do Desenrola.
    
    - dim_data_base e dim_uf: DIMENSÕES COMPARTILHADAS com SCR.
      Atualiza com novos valores do Desenrola via left anti join (idempotente).
    - dim_conglomerado: dimensão específica do Desenrola.
    """
    print("\n=== INICIANDO CRIAÇÃO DAS DIMENSÕES GOLD — DESENROLA ===")
    df_silver = spark.table(TABELA_SILVER_DESENROLA)
    
    # 1. dim_data_base (COMPARTILHADA com SCR) — adiciona datas do Desenrola
    df_data_base = (df_silver
        .select(F.col("DATA_BASE").alias("data_base"))
        .distinct()
        .withColumns({
            "ano": F.year(F.col("data_base")),
            "mes": F.month(F.col("data_base")),
        }))
    criar_dimensao_idempotente(DIM_DATA_BASE, "id_data_base", df_data_base, ["data_base"])
    
    # 2. dim_uf (COMPARTILHADA com SCR) — adiciona UFs do Desenrola com região
    garantir_coluna_regiao_dim_uf()
    df_uf = enriquecer_dim_uf_com_regiao(df_silver.select(F.col("UF").alias("uf")).distinct())
    criar_dimensao_idempotente(DIM_UF, "id_uf", df_uf, ["uf"])
    
    # 3. dim_conglomerado (ESPECÍFICA do Desenrola)
    df_conglomerado = (df_silver
        .select(
            F.col("COD_CONGLOMERADO_FINANCEIRO").alias("cod_conglomerado_financeiro"),
            F.col("NOME_CONGLOMERADO_FINANCEIRO").alias("nome_conglomerado_financeiro"),
        )
        # Garante 1 nome por código (alguns códigos têm nomes variados na silver)
        .dropDuplicates(["cod_conglomerado_financeiro"]))
    criar_dimensao_idempotente(DIM_CONGLOMERADO, "id_conglomerado", df_conglomerado,
        ["cod_conglomerado_financeiro"])
    
    print("\n=== DIMENSÕES GOLD — DESENROLA FINALIZADAS ===")

# Executa a criação das dimensões
criar_dimensoes_desenrola()

# Amostra da dimensão conglomerado
print("\n--- Amostra dim_conglomerado ---")
display(spark.table(DIM_CONGLOMERADO).limit(10))

In [0]:
def criar_fato_desenrola(tabela_silver, tabela_fato):
    """Cria a tabela fato de negociação do Desenrola a partir da silver + dimensões.
    
    - Joins com dim_data_base, dim_uf (compartilhadas) e dim_conglomerado (específica)
    - Converte VOLUME_OPERACOES de string (vírgula decimal) para decimal(18,2)
    - Agrega medidas por (id_data_base, id_uf, id_conglomerado, tipo_desenrola)
    - Valida integridade referencial (FKs não nulas) e duplicação ao final
    """
    print(f"\n=== CRIANDO TABELA FATO: {tabela_fato} ===")
    
    df_silver = spark.table(tabela_silver)
    
    # Joins com dimensões usando aliases para evitar ambiguidade de colunas
    df_gold = (df_silver.alias("s")
        .join(spark.table(DIM_DATA_BASE).alias("ddb"),
              F.col("s.DATA_BASE") == F.col("ddb.data_base"), "left")
        .join(spark.table(DIM_UF).alias("du"),
              F.col("s.UF") == F.col("du.uf"), "left")
        .join(spark.table(DIM_CONGLOMERADO).alias("dc"),
              F.col("s.COD_CONGLOMERADO_FINANCEIRO") == F.col("dc.cod_conglomerado_financeiro"), "left")
        .select(
            F.col("ddb.id_data_base"),
            F.col("du.id_uf"),
            F.col("dc.id_conglomerado"),
            F.col("s.TIPO_DESENROLA").alias("tipo_desenrola"),
            F.col("s.NUMERO_OPERACOES").alias("NUMERO_OPERACOES"),
            # VOLUME_OPERACOES: string com vírgula decimal -> decimal(18,2)
            F.regexp_replace(F.col("s.VOLUME_OPERACOES"), ",", ".")
              .cast("decimal(18,2)").alias("VOLUME_OPERACOES"),
        )
    )
    
    # Validação de integridade referencial: nenhum FK deve ser nulo
    fk_nulos = df_gold.filter(
        F.col("id_data_base").isNull() | F.col("id_uf").isNull() |
        F.col("id_conglomerado").isNull()
    ).count()
    
    if fk_nulos > 0:
        print(f"[Gold] AVISO: {fk_nulos} registros com FK nula — verificar dimensões.")
    else:
        print(f"[Gold] OK: Integridade referencial validada — todas as FKs preenchidas.")
    
    # Agregação por chaves dimensionais + tipo_desenrola (elimina duplicação por design)
    df_fct = (df_gold
        .groupBy("id_data_base", "id_uf", "id_conglomerado", "tipo_desenrola")
        .agg(
            F.sum("NUMERO_OPERACOES").alias("numero_operacoes"),
            F.sum("VOLUME_OPERACOES").alias("volume_operacoes"),
        )
    )
    
    # Escrita idempotente: overwrite (a agregação é determinística a partir da silver)
    (df_fct.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(tabela_fato))
    
    # Validação de duplicação: a combinação de todas as FKs + tipo_desenrola deve ser única
    total = spark.table(tabela_fato).count()
    distinto = (spark.table(tabela_fato)
        .select("id_data_base", "id_uf", "id_conglomerado", "tipo_desenrola")
        .distinct().count())
    
    if total != distinto:
        print(f"[Gold] AVISO: {total - distinto} duplicatas na tabela fato!")
    else:
        print(f"[Gold] OK: {total} registros únicos sem duplicação.")
    
    print(f"[Gold] Fato {tabela_fato} criada com {total} registros.")

# Executa a criação da fato
criar_fato_desenrola(TABELA_SILVER_DESENROLA, TABELA_FCT_DESENROLA)

# Amostra da fato criada
print("\n--- Amostra fct_desenrola_negociacao ---")
display(spark.table(TABELA_FCT_DESENROLA).limit(10))

   
### Visualizações — Camada Gold

Gráficos analíticos gerados a partir das tabelas fato da camada Gold.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: endividamento de longo prazo (vencido acima de 90 dias) por modalidade em 2025
df_endiv_longo = spark.sql("""
    SELECT dm.modalidade,
           CAST(SUM(f.vencido_acima_de_90_dias) AS DOUBLE) AS total_vencido_acima_90
    FROM mvp_pos_puc.default.fct_taxa_de_endividamento f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_modalidade dm ON f.id_modalidade = dm.id_modalidade
    WHERE ddb.ano = 2025
    GROUP BY dm.modalidade
    ORDER BY total_vencido_acima_90 DESC
""")

pdf_endiv = df_endiv_longo.toPandas()

# Calcula percentuais
total_geral = pdf_endiv['total_vencido_acima_90'].sum()
pdf_endiv['percentual'] = (pdf_endiv['total_vencido_acima_90'] / total_geral * 100).round(2)

# Agrupa categorias menores que 2% em "Outros"
pdf_plot = pdf_endiv[pdf_endiv['percentual'] >= 2.0].copy()
outros = pdf_endiv[pdf_endiv['percentual'] < 2.0]
if len(outros) > 0:
    outros_row = {
        'modalidade': 'Outros',
        'total_vencido_acima_90': outros['total_vencido_acima_90'].sum(),
        'percentual': outros['percentual'].sum().round(2)
    }
    pdf_plot = pdf_plot._append(outros_row, ignore_index=True)

print(f"Total geral de endividamento de longo prazo (acima de 90 dias) em 2025: R$ {total_geral:,.2f}")
print(f"\nMaiores modalidades:")
for _, row in pdf_endiv.head(5).iterrows():
    print(f"  {row['modalidade']}: R$ {row['total_vencido_acima_90']:,.2f} ({row['percentual']}%)")

# Gráfico de pizza
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set3(range(len(pdf_plot)))
wedges, texts, autotexts = ax.pie(
    pdf_plot['total_vencido_acima_90'],
    labels=pdf_plot['modalidade'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    pctdistance=0.85,
    textprops={'fontsize': 10}
)

ax.set_title('Endividamento de Longo Prazo (Acima de 90 Dias) por Modalidade — 2025',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: endividamento de longo prazo (vencido acima de 90 dias) por modalidade e tipo de cliente em 2025
df_endiv_tipo = spark.sql("""
    SELECT dtc.cliente AS tipo_cliente,
           dm.modalidade,
           CAST(SUM(f.vencido_acima_de_90_dias) AS DOUBLE) AS total_vencido_acima_90
    FROM mvp_pos_puc.default.fct_taxa_de_endividamento f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_modalidade dm ON f.id_modalidade = dm.id_modalidade
    JOIN mvp_pos_puc.default.dim_tipo_cliente dtc ON f.id_tipo_cliente = dtc.id_tipo_cliente
    WHERE ddb.ano = 2025
    GROUP BY dtc.cliente, dm.modalidade
    ORDER BY dtc.cliente, total_vencido_acima_90 DESC
""")

pdf_endiv_tipo = df_endiv_tipo.toPandas()

# Função para preparar dados do gráfico (agrupa categorias < 2% em "Outros")
def preparar_dados(df_filtrado):
    total = df_filtrado['total_vencido_acima_90'].sum()
    df_filtrado = df_filtrado.copy()
    df_filtrado['percentual'] = (df_filtrado['total_vencido_acima_90'] / total * 100).round(2)
    df_principais = df_filtrado[df_filtrado['percentual'] >= 2.0].copy()
    outros = df_filtrado[df_filtrado['percentual'] < 2.0]
    if len(outros) > 0:
        outros_row = {
            'tipo_cliente': df_filtrado['tipo_cliente'].iloc[0],
            'modalidade': 'Outros',
            'total_vencido_acima_90': outros['total_vencido_acima_90'].sum(),
            'percentual': round(outros['percentual'].sum(), 2)
        }
        df_principais = df_principais._append(outros_row, ignore_index=True)
    return df_principais, total

# Prepara dados para PF e PJ
df_pf, total_pf = preparar_dados(pdf_endiv_tipo[pdf_endiv_tipo['tipo_cliente'] == 'PF'])
df_pj, total_pj = preparar_dados(pdf_endiv_tipo[pdf_endiv_tipo['tipo_cliente'] == 'PJ'])

print(f"Total PF — Endividamento longo prazo (acima de 90 dias) em 2025: R$ {total_pf:,.2f}")
print(f"Total PJ — Endividamento longo prazo (acima de 90 dias) em 2025: R$ {total_pj:,.2f}")

# Cria os dois gráficos de pizza lado a lado
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

colors_pf = plt.cm.Set3(range(len(df_pf)))
colors_pj = plt.cm.Set2(range(len(df_pj)))

# Gráfico PF
wedges_pf, texts_pf, autotexts_pf = axes[0].pie(
    df_pf['total_vencido_acima_90'],
    labels=df_pf['modalidade'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors_pf,
    pctdistance=0.82,
    textprops={'fontsize': 9}
)
axes[0].set_title(f'Pessoa Física (PF)\nTotal: R$ {total_pf/1e9:.2f} bi',
                   fontsize=13, fontweight='bold', pad=15)

# Gráfico PJ
wedges_pj, texts_pj, autotexts_pj = axes[1].pie(
    df_pj['total_vencido_acima_90'],
    labels=df_pj['modalidade'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors_pj,
    pctdistance=0.82,
    textprops={'fontsize': 9}
)
axes[1].set_title(f'Pessoa Jurídica (PJ)\nTotal: R$ {total_pj/1e9:.2f} bi',
                   fontsize=13, fontweight='bold', pad=15)

fig.suptitle('Endividamento de Longo Prazo (Acima de 90 Dias) por Modalidade e Tipo de Cliente — 2025',
             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: conglomerados que mais realizaram negociações no Desenrola em 2025
df_conglomerados = spark.sql("""
    SELECT dc.nome_conglomerado_financeiro AS conglomerado,
           SUM(f.numero_operacoes) AS total_operacoes
    FROM mvp_pos_puc.default.fct_desenrola_negociacao f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_conglomerado dc ON f.id_conglomerado = dc.id_conglomerado
    WHERE ddb.ano = 2025
    GROUP BY dc.nome_conglomerado_financeiro
    ORDER BY total_operacoes DESC
""")

pdf_congl = df_conglomerados.toPandas()

# Top 10 conglomerados + "Outros"
top_n = 10
total_geral = pdf_congl['total_operacoes'].sum()

pdf_top = pdf_congl.head(top_n).copy()
outros_total = pdf_congl.iloc[top_n:]['total_operacoes'].sum()
if len(pdf_congl) > top_n:
    outros_row = {
        'conglomerado': f'Outros ({len(pdf_congl) - top_n} conglomerados)',
        'total_operacoes': outros_total
    }
    pdf_top = pdf_top._append(outros_row, ignore_index=True)

pdf_top['percentual'] = (pdf_top['total_operacoes'] / total_geral * 100).round(2)

print(f"Total de negociações no Desenrola em 2025: {total_geral:,.0f}")
print(f"Total de conglomerados distintos: {len(pdf_congl)}")
print(f"\nTop 10 conglomerados:")
for _, row in pdf_congl.head(10).iterrows():
    pct = (row['total_operacoes'] / total_geral * 100)
    print(f"  {row['conglomerado']}: {row['total_operacoes']:,.0f} ({pct:.2f}%)")

# Gráfico de pizza
fig, ax = plt.subplots(figsize=(12, 8))

colors = plt.cm.tab20(range(len(pdf_top)))
wedges, texts, autotexts = ax.pie(
    pdf_top['total_operacoes'],
    labels=pdf_top['conglomerado'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    pctdistance=0.85,
    textprops={'fontsize': 9}
)

ax.set_title('Conglomerados que Mais Realizaram Negociações — Desenrola Brasil 2025',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: taxa de endividamento ao longo de 2025 (por mês)
# Taxa de endividamento = carteira_vencida / carteira_ativa
# Taxa de inadimplência = carteira_inadimplencia / carteira_ativa
df_taxa_endiv = spark.sql("""
    SELECT ddb.mes,
           CAST(SUM(f.carteira_ativa) AS DOUBLE) AS carteira_ativa,
           CAST(SUM(f.carteira_vencida) AS DOUBLE) AS carteira_vencida,
           CAST(SUM(f.carteira_inadimplencia) AS DOUBLE) AS carteira_inadimplencia
    FROM mvp_pos_puc.default.fct_taxa_de_endividamento f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    WHERE ddb.ano = 2025
    GROUP BY ddb.mes
    ORDER BY ddb.mes
""").toPandas()

# Calcula as taxas
df_taxa_endiv['taxa_endividamento'] = (df_taxa_endiv['carteira_vencida'] / df_taxa_endiv['carteira_ativa']) * 100
df_taxa_endiv['taxa_inadimplencia'] = (df_taxa_endiv['carteira_inadimplencia'] / df_taxa_endiv['carteira_ativa']) * 100

print("Taxa de Endividamento ao longo de 2025:")
for _, row in df_taxa_endiv.iterrows():
    print(f"  Mês {int(row['mes']):02d}: Endividamento {row['taxa_endividamento']:.2f}% | Inadimplência {row['taxa_inadimplencia']:.2f}%")

# Gráfico de linhas
fig, ax1 = plt.subplots(figsize=(14, 7))

meses = df_taxa_endiv['mes'].astype(int)
nomes_meses = [f'{m:02d}/2025' for m in meses]

ax1.plot(nomes_meses, df_taxa_endiv['taxa_endividamento'], marker='o', linewidth=2.5,
         color='#e74c3c', label='Taxa de Endividamento (Vencida/Ativa)', markersize=8)
ax1.plot(nomes_meses, df_taxa_endiv['taxa_inadimplencia'], marker='s', linewidth=2.5,
         color='#2980b9', label='Taxa de Inadimplência (Inadimplência/Ativa)', markersize=8)

ax1.set_title('Taxa de Endividamento ao Longo de 2025 — SCR', fontsize=15, fontweight='bold', pad=15)
ax1.set_xlabel('Mês', fontsize=12)
ax1.set_ylabel('Taxa (%)', fontsize=12)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax1.legend(fontsize=11, loc='upper left')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.tick_params(axis='x', rotation=45)

# Eixo secundário: volume da carteira ativa
ax2 = ax1.twinx()
ax2.bar(nomes_meses, df_taxa_endiv['carteira_ativa'] / 1e9, alpha=0.15, color='#2ecc71', width=0.5)
ax2.set_ylabel('Carteira Ativa (R$ bilhões)', fontsize=12, color='#27ae60')
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'R$ {x:.0f} bi'))
ax2.tick_params(axis='y', labelcolor='#27ae60')

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: inadimplência por mês e região em 2025
df_inad_2025 = spark.sql("""
    SELECT ddb.mes, du.regiao,
           CAST(SUM(f.carteira_inadimplencia) AS DOUBLE) AS inadimplencia
    FROM mvp_pos_puc.default.fct_taxa_de_endividamento f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_uf du ON f.id_uf = du.id_uf
    WHERE ddb.ano = 2025
    GROUP BY ddb.mes, du.regiao
    ORDER BY ddb.mes
""").toPandas()

# Pivot: região nas colunas, mês no eixo X
pivot_inad = df_inad_2025.pivot(index='mes', columns='regiao', values='inadimplencia').fillna(0)

print("Inadimplência por Região ao longo de 2025 (R$ bilhões):")
for mes in sorted(pivot_inad.index):
    print(f"  Mês {int(mes):02d}: " + " | ".join(
        [f"{reg}: R$ {pivot_inad.loc[mes, reg]/1e9:.2f} bi" for reg in pivot_inad.columns]
    ))

# Gráfico de linhas — uma linha por região
fig, ax = plt.subplots(figsize=(14, 7))

nomes_meses = [f'{int(m):02d}/2025' for m in sorted(pivot_inad.index)]

for regiao in pivot_inad.columns:
    ax.plot(nomes_meses, [pivot_inad.loc[m, regiao] for m in sorted(pivot_inad.index)],
            marker='o', linewidth=2, markersize=6, label=regiao)

ax.set_title('Inadimplência por Região ao Longo de 2025 — SCR', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Mês', fontsize=12)
ax.set_ylabel('Inadimplência (R$)', fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'R$ {x/1e9:.1f} bi'))
ax.legend(title='Região', fontsize=11, title_fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: número de negociações do Desenrola por mês e região em 2025
df_neg_2025 = spark.sql("""
    SELECT ddb.mes, du.regiao,
           CAST(SUM(f.numero_operacoes) AS DOUBLE) AS numero_operacoes
    FROM mvp_pos_puc.default.fct_desenrola_negociacao f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_uf du ON f.id_uf = du.id_uf
    WHERE ddb.ano = 2025
    GROUP BY ddb.mes, du.regiao
    ORDER BY ddb.mes
""").toPandas()

# Pivot: região nas colunas, mês no eixo X
pivot_neg = df_neg_2025.pivot(index='mes', columns='regiao', values='numero_operacoes').fillna(0)

print("Negociações do Desenrola por Região ao longo de 2025:")
for mes in sorted(pivot_neg.index):
    print(f"  Mês {int(mes):02d}: " + " | ".join(
        [f"{reg}: {pivot_neg.loc[mes, reg]:,.0f}" for reg in pivot_neg.columns]
    ))

# Gráfico de linhas — uma linha por região
fig, ax = plt.subplots(figsize=(14, 7))

nomes_meses = [f'{int(m):02d}/2025' for m in sorted(pivot_neg.index)]

for regiao in pivot_neg.columns:
    ax.plot(nomes_meses, [pivot_neg.loc[m, regiao] for m in sorted(pivot_neg.index)],
            marker='o', linewidth=2, markersize=6, label=regiao)

ax.set_title('Negociações do Desenrola por Região ao Longo de 2025', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Mês', fontsize=12)
ax.set_ylabel('Número de Negociações', fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))
ax.legend(title='Região', fontsize=11, title_fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: Empréstimos em 2025 — carteira ativa vs inadimplência
df_emp_2025 = spark.sql("""
    SELECT
        CAST(SUM(f.carteira_ativa) AS DOUBLE) AS carteira_ativa,
        CAST(SUM(f.carteira_inadimplencia) AS DOUBLE) AS inadimplencia
    FROM mvp_pos_puc.default.fct_taxa_de_endividamento f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_modalidade dm ON f.id_modalidade = dm.id_modalidade
    WHERE ddb.ano = 2025
      AND dm.modalidade = 'Empréstimos'
""").toPandas()

carteira_ativa = df_emp_2025['carteira_ativa'].iloc[0]
inadimplencia = df_emp_2025['inadimplencia'].iloc[0]
nao_inadimplente = carteira_ativa - inadimplencia
taxa_inad = (inadimplencia / carteira_ativa) * 100

print(f"Empréstimos — 2025")
print(f"  Carteira Ativa (total concedido): R$ {carteira_ativa:,.2f}")
print(f"  Inadimplência:                     R$ {inadimplencia:,.2f} ({taxa_inad:.2f}%)")
print(f"  Não inadimplente (em dia):          R$ {nao_inadimplente:,.2f} ({100 - taxa_inad:.2f}%)")

# Gráfico de pizza: Inadimplente vs Não Inadimplente
fig, ax = plt.subplots(figsize=(10, 8))

valores = [nao_inadimplente, inadimplencia]
labels = [
    f'Em Dia\nR$ {nao_inadimplente/1e9:.2f} bi\n({100 - taxa_inad:.2f}%)',
    f'Inadimplente\nR$ {inadimplencia/1e9:.2f} bi\n({taxa_inad:.2f}%)'
]
colors = ['#2ecc71', '#e74c3c']

wedges, texts, autotexts = ax.pie(
    valores,
    labels=labels,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    pctdistance=0.75,
    textprops={'fontsize': 12, 'fontweight': 'bold'},
    explode=(0, 0.05)
)

ax.set_title(f'Empréstimos 2025: Inadimplência vs Carteira em Dia\nTotal da Carteira Ativa: R$ {carteira_ativa/1e9:.2f} bilhões',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: volume de operações do Desenrola por tipo_desenrola em 2025
df_vol_tipo = spark.sql("""
    SELECT f.tipo_desenrola,
           CAST(SUM(f.volume_operacoes) AS DOUBLE) AS volume_total
    FROM mvp_pos_puc.default.fct_desenrola_negociacao f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    WHERE ddb.ano = 2025
    GROUP BY f.tipo_desenrola
    ORDER BY volume_total DESC
""").toPandas()

total_geral = df_vol_tipo['volume_total'].sum()
df_vol_tipo['percentual'] = (df_vol_tipo['volume_total'] / total_geral * 100).round(2)

print(f"Volume total de operações do Desenrola em 2025: R$ {total_geral:,.2f}")
print(f"\nPor tipo:")
for _, row in df_vol_tipo.iterrows():
    print(f"  {row['tipo_desenrola']}: R$ {row['volume_total']:,.2f} ({row['percentual']}%)")

# Gráfico de pizza
fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.Set2(range(len(df_vol_tipo)))
wedges, texts, autotexts = ax.pie(
    df_vol_tipo['volume_total'],
    labels=df_vol_tipo['tipo_desenrola'],
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    pctdistance=0.80,
    textprops={'fontsize': 10}
)

ax.set_title(f'Volume de Operações do Desenrola por Tipo — 2025\nTotal: R$ {total_geral/1e9:.2f} bilhões',
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: carteira de inadimplência por mês e região (2023–2025)
df_inadimplencia = spark.sql("""
    SELECT ddb.ano, ddb.mes, du.regiao,
           CAST(SUM(f.carteira_inadimplencia) AS DOUBLE) AS inadimplencia
    FROM mvp_pos_puc.default.fct_taxa_de_endividamento f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_uf du ON f.id_uf = du.id_uf
    WHERE ddb.ano IN (2023, 2024, 2025)
    GROUP BY ddb.ano, ddb.mes, du.regiao
""").toPandas()

# Chave ano-mês
df_inadimplencia['ano_mes'] = df_inadimplencia['ano'].astype(str) + '-' + df_inadimplencia['mes'].astype(str).str.zfill(2)

# Pivot: região nas colunas, ano-mês no eixo X
pivot_df = df_inadimplencia.pivot(index='ano_mes', columns='regiao', values='inadimplencia').fillna(0)

fig, ax = plt.subplots(figsize=(16, 7))
pivot_df.plot(kind='bar', stacked=False, ax=ax, colormap='Set1', width=0.8)

ax.set_title('Carteira de Inadimplência por Região ao Longo dos Meses — 2023 a 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano-Mês', fontsize=12)
ax.set_ylabel('Inadimplência (R$)', fontsize=12)
ax.legend(title='Região', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'R$ {x/1e9:.1f} Bi'))
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Consulta: número de negociações por mês e região (2023–2025)
df_negociacoes = spark.sql("""
    SELECT ddb.ano, ddb.mes, du.regiao,
           CAST(SUM(f.numero_operacoes) AS DOUBLE) AS numero_operacoes
    FROM mvp_pos_puc.default.fct_desenrola_negociacao f
    JOIN mvp_pos_puc.default.dim_data_base ddb ON f.id_data_base = ddb.id_data_base
    JOIN mvp_pos_puc.default.dim_uf du ON f.id_uf = du.id_uf
    WHERE ddb.ano IN (2023, 2024, 2025)
    GROUP BY ddb.ano, ddb.mes, du.regiao
""").toPandas()

# Chave ano-mês
df_negociacoes['ano_mes'] = df_negociacoes['ano'].astype(str) + '-' + df_negociacoes['mes'].astype(str).str.zfill(2)

# Pivot: região nas colunas, ano-mês no eixo X
pivot_df = df_negociacoes.pivot(index='ano_mes', columns='regiao', values='numero_operacoes').fillna(0)

fig, ax = plt.subplots(figsize=(16, 7))
pivot_df.plot(kind='bar', stacked=False, ax=ax, colormap='Set1', width=0.8)

ax.set_title('Número de Negociações do Desenrola por Região ao Longo dos Meses — 2023 a 2025', fontsize=14, fontweight='bold')
ax.set_xlabel('Ano-Mês', fontsize=12)
ax.set_ylabel('Número de Negociações', fontsize=12)
ax.legend(title='Região', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Bloco Executável Principal (Orquestração)

In [0]:
try:
    # Garante a existência do diretório temporário raiz limpo no escopo
    os.makedirs(DIRETORIO_RAIZ_TMP, exist_ok=True)
    
    # 1. Carga Bronze — SCR (Parametrizado via Widget)
    executar_pipeline_scr(QUANTIDADE_ANOS_SCR, TABELA_SCR)
    
    # 2. Carga Bronze — Desenrola Brasil (Base única consolidada)
    executar_pipeline_desenrola(URL_DESENROLA, TABELA_DESENROLA)
    
    # 2.5. Qualidade de Dados - Tratamento de nulos e deteccao de outliers (apos Bronze, antes da Silver)
    for tabela in TABELAS_BRONZE_QUALIDADE:
        tratar_nulos_e_outliers(tabela)
    
    # 3. Transformação Silver — SCR (após carga bronze)
    criar_silver_scr(TABELA_BRONZE_SCR, TABELA_SILVER_SCR)
    
    # 4. Transformação Silver — Desenrola (após carga bronze)
    criar_silver_desenrola(TABELA_BRONZE_DESENROLA, TABELA_SILVER_DESENROLA)
    
    # 5. Camada Gold — Dimensões SCR (a partir da silver)
    criar_dimensoes_scr()
    
    # 6. Camada Gold — Fato SCR (a partir da silver + dimensões)
    criar_fato_endividamento(TABELA_SILVER_SCR, TABELA_FCT_ENDIVIDAMENTO)
    
    # 7. Camada Gold — Dimensões Desenrola (compartilha dim_data_base e dim_uf com SCR)
    criar_dimensoes_desenrola()
    
    # 8. Camada Gold — Fato Desenrola (a partir da silver + dimensões)
    criar_fato_desenrola(TABELA_SILVER_DESENROLA, TABELA_FCT_DESENROLA)
    
    print("\n=== EXECUÇÃO DE TODAS AS PIPELINES FINALIZADA COM SUCESSO ===")
    
except Exception as e:
    print(f"\n[Falha Crítica] Erro na orquestração central das tabelas: {e}")
    
finally:
    # Limpeza preventiva total de resíduos temporários do Workspace Databricks
    if os.path.exists(DIRETORIO_RAIZ_TMP):
        shutil.rmtree(DIRETORIO_RAIZ_TMP)
        print("[Limpeza] Diretório raiz temporário de controle removido com sucesso.")

# Catálogo de Dados — Pipeline Unificada BCB

**Schema:** `mvp_pos_puc.default`  
**Total de tabelas:** 13  
**Fonte de dados:** Dados Abertos do Banco Central do Brasil (https://dadosabertos.bcb.gov.br)

---

## 🦥 Camada Bronze — Ingestão Raw

Dados brutos ingeridos diretamente da fonte BCB, com metadados de auditoria injetados automaticamente (`nome_arquivo_origem`, `data_processamento`, `origem_carga`) e coluna `id` auto-incremental para unicidade.

---

### 1. `brz_bcb_scr`

**Descrição:** Dados brutos do SCR (Sistema de Informações de Crédito) — versão 2 do SCR.Data. Ingeridos a partir de arquivos ZIP anuais baixados de `https://www.bcb.gov.br/pda/desig/scrdata_{ANO}.zip`. Particionada por `ano_carga` com idempotência via `replaceWhere`.  
**Registros:** 14.194.430  
**Colunas:** 29  
**Granularidade:** data_base × uf × segmento × cliente × porte × modalidade × submodalidade × origem × cnae_ocupacao  
**Forma de carga:** Download de ZIP anual → extração → leitura via Spark CSV (`sep=;`) → `replaceWhere` por partição `ano_carga`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `data_base` | DATE | SIM | Data de referência da carteira (último dia útil do mês) |
| 2 | `uf` | STRING | SIM | Unidade da Federação |
| 3 | `segmento` | STRING | SIM | Segmento da operação (ex: PF, PJ, Rural) |
| 4 | `cliente` | STRING | SIM | Tipo de cliente (ex: PF, PJ) |
| 5 | `cnae_ocupacao` | STRING | SIM | Código CNAE da ocupação |
| 6 | `porte` | STRING | SIM | Porte do cliente (ex: até 2 SM, até 20 mil, etc.) |
| 7 | `modalidade` | STRING | SIM | Modalidade de crédito (ex: Empréstimos, Financiamentos) |
| 8 | `submodalidade` | STRING | SIM | Submodalidade da operação |
| 9 | `origem` | STRING | SIM | Origem dos recursos |
| 10 | `indexador` | STRING | SIM | Indexador da operação (removido na camada Silver) |
| 11 | `numero_de_operacoes` | INT | SIM | Quantidade de operações |
| 12 | `a_vencer_ate_90_dias` | STRING | SIM | Saldo a vencer até 90 dias (formato brasileiro, vírgula decimal) |
| 13 | `a_vencer_de_91_ate_360_dias` | STRING | SIM | Saldo a vencer de 91 a 360 dias |
| 14 | `a_vencer_de_361_ate_1080_dias` | STRING | SIM | Saldo a vencer de 361 a 1080 dias |
| 15 | `a_vencer_de_1081_ate_1800_dias` | STRING | SIM | Saldo a vencer de 1081 a 1800 dias |
| 16 | `a_vencer_de_1801_ate_5400_dias` | STRING | SIM | Saldo a vencer de 1801 a 5400 dias |
| 17 | `a_vencer_acima_de_5400_dias` | STRING | SIM | Saldo a vencer acima de 5400 dias |
| 18 | `carteira_a_vencer` | STRING | SIM | Total da carteira a vencer |
| 19 | `vencido_de_15_ate_90_dias` | STRING | SIM | Saldo vencido de 15 a 90 dias |
| 20 | `vencido_acima_de_90_dias` | STRING | SIM | Saldo vencido acima de 90 dias |
| 21 | `carteira_vencida` | STRING | SIM | Total da carteira vencida |
| 22 | `carteira_ativa` | STRING | SIM | Total da carteira ativa |
| 23 | `carteira_inadimplencia` | STRING | SIM | Saldo em inadimplência |
| 24 | `ativo_problematico` | STRING | SIM | Ativo problemático |
| 25 | `nome_arquivo_origem` | STRING | SIM | Metadado de auditoria — caminho do arquivo de origem |
| 26 | `data_processamento` | TIMESTAMP | SIM | Metadado de auditoria — timestamp da carga |
| 27 | `origem_carga` | STRING | SIM | Metadado de auditoria — origem fixa `BCB_PDA` |
| 28 | `ano_carga` | INT | SIM | Ano de referência (chave de partição) |
| 29 | `id` | INT | SIM | Surrogate key auto-incremental |

---

### 2. `brz_bcb_desenrola`

**Descrição:** Dados brutos do programa Desenrola Brasil (PF e Pequenos Negócios). Ingeridos a partir de CSV único e consolidado de `https://www.bcb.gov.br/pda/desig/desenrola/dados_desenrola.csv`. Sobrescrita completa idempotente a cada carga.  
**Registros:** 12.751  
**Colunas:** 11  
**Granularidade:** data_base × tipo_desenrola × unidade_federacao × cod_conglomerado_financeiro  
**Forma de carga:** Download de CSV Único → leitura via pandas → `mode=overwrite` completo

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `DATA_BASE` | LONG | SIM | Data de referência no formato YYYYMM (ex: 202312) |
| 2 | `TIPO_DESENROLA` | LONG | SIM | Tipo do programa: 1=PF_ate_2_salarios, 2=PF_ate_20_mil, 3=PJ_faturamento_ate_4_8_mm |
| 3 | `UNIDADE_FEDERACAO` | STRING | SIM | Unidade da Federação (renomeada para `UF` na camada Silver) |
| 4 | `COD_CONGLOMERADO_FINANCEIRO` | LONG | SIM | Código do conglomerado financeiro |
| 5 | `NOME_CONGLOMERADO_FINANCEIRO` | STRING | SIM | Nome do conglomerado financeiro |
| 6 | `NUMERO_OPERACOES` | LONG | SIM | Quantidade de operações de renegociação |
| 7 | `VOLUME_OPERACOES` | STRING | SIM | Volume financeiro das operações (vírgula decimal) |
| 8 | `nome_arquivo_origem` | STRING | SIM | Metadado de auditoria — caminho do arquivo de origem |
| 9 | `data_processamento` | TIMESTAMP | SIM | Metadado de auditoria — timestamp da carga |
| 10 | `origem_carga` | STRING | SIM | Metadado de auditoria — origem fixa `BCB_PDA` |
| 11 | `id` | INT | SIM | Surrogate key auto-incremental |

---

## 🧧 Camada Silver — Transformação e Limpeza

Dados limpos, tipados e padronizados. Remoção de metadados de auditoria, conversão de tipos, padronização de nomes de colunas para UPPERCASE e tratamento de formatos brasileiros (vírgula decimal → ponto).

---

### 3. `slv_bcb_scr`

**Descrição:** Tabela silver do SCR. Remove `indexador`, `nome_arquivo_origem`, `data_processamento`, `origem_carga` e `ano_carga`. Converte 14 colunas de saldo de `STRING` (vírgula decimal) para `DECIMAL(18,2)`. Padroniza nomes para UPPERCASE.  
**Registros:** 14.194.430  
**Colunas:** 24  
**Granularidade:** DATA_BASE × UF × SEGMENTO × CLIENTE × PORTE × MODALIDADE × SUBMODALIDADE × ORIGEM × CNAE_OCUPACAO  
**Transformações:** `regexp_replace(',', '.')` → `cast('decimal(18,2)')` nas colunas de saldo

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `DATA_BASE` | DATE | SIM | Data de referência da carteira |
| 2 | `UF` | STRING | SIM | Unidade da Federação |
| 3 | `SEGMENTO` | STRING | SIM | Segmento da operação |
| 4 | `CLIENTE` | STRING | SIM | Tipo de cliente (PF/PJ) |
| 5 | `CNAE_OCUPACAO` | STRING | SIM | Código CNAE da ocupação |
| 6 | `PORTE` | STRING | SIM | Porte do cliente |
| 7 | `MODALIDADE` | STRING | SIM | Modalidade de crédito |
| 8 | `SUBMODALIDADE` | STRING | SIM | Submodalidade da operação |
| 9 | `ORIGEM` | STRING | SIM | Origem dos recursos |
| 10 | `NUMERO_DE_OPERACOES` | DECIMAL(18,2) | SIM | Quantidade de operações |
| 11 | `A_VENCER_ATE_90_DIAS` | DECIMAL(18,2) | SIM | Saldo a vencer até 90 dias |
| 12 | `A_VENCER_DE_91_ATE_360_DIAS` | DECIMAL(18,2) | SIM | Saldo a vencer de 91 a 360 dias |
| 13 | `A_VENCER_DE_361_ATE_1080_DIAS` | DECIMAL(18,2) | SIM | Saldo a vencer de 361 a 1080 dias |
| 14 | `A_VENCER_DE_1081_ATE_1800_DIAS` | DECIMAL(18,2) | SIM | Saldo a vencer de 1081 a 1800 dias |
| 15 | `A_VENCER_DE_1801_ATE_5400_DIAS` | DECIMAL(18,2) | SIM | Saldo a vencer de 1801 a 5400 dias |
| 16 | `A_VENCER_ACIMA_DE_5400_DIAS` | DECIMAL(18,2) | SIM | Saldo a vencer acima de 5400 dias |
| 17 | `CARTEIRA_A_VENCER` | DECIMAL(18,2) | SIM | Total da carteira a vencer |
| 18 | `VENCIDO_DE_15_ATE_90_DIAS` | DECIMAL(18,2) | SIM | Saldo vencido de 15 a 90 dias |
| 19 | `VENCIDO_ACIMA_DE_90_DIAS` | DECIMAL(18,2) | SIM | Saldo vencido acima de 90 dias |
| 20 | `CARTEIRA_VENCIDA` | DECIMAL(18,2) | SIM | Total da carteira vencida |
| 21 | `CARTEIRA_ATIVA` | DECIMAL(18,2) | SIM | Total da carteira ativa |
| 22 | `CARTEIRA_INADIMPLENCIA` | DECIMAL(18,2) | SIM | Saldo em inadimplência |
| 23 | `ATIVO_PROBLEMATICO` | DECIMAL(18,2) | SIM | Ativo problemático |
| 24 | `ID` | INT | SIM | Surrogate key (herdado da bronze) |

---

### 4. `slv_bcb_desenrola`

**Descrição:** Tabela silver do Desenrola. Remove `nome_arquivo_origem`, `data_processamento`, `origem_carga`. Converte `DATA_BASE` de `LONG` (YYYYMM) para `DATE` (dia 01 do mês). Mapeia `TIPO_DESENROLA` de inteiro para string descritiva. Renomeia `UNIDADE_FEDERACAO` para `UF`.  
**Registros:** 12.751  
**Colunas:** 8  
**Granularidade:** DATA_BASE × TIPO_DESENROLA × UF × COD_CONGLOMERADO_FINANCEIRO  
**Transformações:** `to_date(cast(string), 'yyyyMM')`, mapeamento `TIPO_DESENROLA` (1→PF_ATE_2_SALARIOS, 2→PF_ATE_20_MIL, 3→PJ_FATURAMENTO_ATE_4_8_MM), rename `UNIDADE_FEDERACAO`→`UF`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `DATA_BASE` | DATE | SIM | Data de referência (convertida de YYYYMM para date) |
| 2 | `TIPO_DESENROLA` | STRING | SIM | Tipo do programa: `PF_ATE_2_SALARIOS`, `PF_ATE_20_MIL`, `PJ_FATURAMENTO_ATE_4_8_MM` |
| 3 | `UF` | STRING | SIM | Unidade da Federação (renomeada de `UNIDADE_FEDERACAO`) |
| 4 | `COD_CONGLOMERADO_FINANCEIRO` | LONG | SIM | Código do conglomerado financeiro |
| 5 | `NOME_CONGLOMERADO_FINANCEIRO` | STRING | SIM | Nome do conglomerado financeiro |
| 6 | `NUMERO_OPERACOES` | LONG | SIM | Quantidade de operações de renegociação |
| 7 | `VOLUME_OPERACOES` | STRING | SIM | Volume financeiro das operações (vírgula decimal, convertido na camada Gold) |
| 8 | `ID` | INT | SIM | Surrogate key (herdado da bronze) |

---

## 🦫 Camada Gold — Modelo Dimensional

Modelo dimensional (Star Schema) com tabelas fato agregadas e tabelas dimensão com surrogate keys sequenciais. Dimensões são idempotentes via *left anti join* (novos registros recebem `max(id) + row_number()`). Fatos agregam medidas por combinação de chaves dimensionais (`GROUP BY`).

### Dimensões

---

### 5. `dim_data_base`

**Descrição:** Dimensão temporal compartilhada entre SCR e Desenrola. Uma linha por data_base distinta encontrada em ambas as fontes.  
**Registros:** 84  
**Colunas:** 4  
**Chave natural:** `data_base`  
**Surrogate key:** `id_data_base`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `data_base` | DATE | SIM | Data de referência (chave natural) |
| 2 | `ano` | INT | SIM | Ano extraído de `data_base` |
| 3 | `mes` | INT | SIM | Mês extraído de `data_base` (1–12) |
| 4 | `id_data_base` | INT | SIM | Surrogate key sequencial |

---

### 6. `dim_uf`

**Descrição:** Dimensão geográfica compartilhada entre SCR e Desenrola. Uma linha por UF. Enriquecida com mapeamento UF → região do Brasil via broadcast.  
**Registros:** 27  
**Colunas:** 3  
**Chave natural:** `uf`  
**Surrogate key:** `id_uf`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `uf` | STRING | SIM | Sigla da Unidade da Federação (chave natural) |
| 2 | `regiao` | STRING | SIM | Região do Brasil (Norte, Nordeste, Centro-Oeste, Sudeste, Sul) |
| 3 | `id_uf` | INT | SIM | Surrogate key sequencial |

---

### 7. `dim_segmento`

**Descrição:** Dimensão de segmento econômico do SCR.  
**Registros:** 8  
**Colunas:** 2  
**Chave natural:** `segmento`  
**Surrogate key:** `id_segmento`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `segmento` | STRING | SIM | Nome do segmento (chave natural) |
| 2 | `id_segmento` | INT | SIM | Surrogate key sequencial |

---

### 8. `dim_tipo_cliente`

**Descrição:** Dimensão de tipo e porte do cliente do SCR. Combinação de `cliente` + `porte` como chave natural composta.  
**Registros:** 14  
**Colunas:** 3  
**Chave natural:** `cliente` + `porte`  
**Surrogate key:** `id_tipo_cliente`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `cliente` | STRING | SIM | Tipo de cliente (PF/PJ) (parte da chave natural) |
| 2 | `porte` | STRING | SIM | Porte do cliente (parte da chave natural) |
| 3 | `id_tipo_cliente` | INT | SIM | Surrogate key sequencial |

---

### 9. `dim_ocupacao`

**Descrição:** Dimensão de ocupação (CNAE) do SCR.  
**Registros:** 30  
**Colunas:** 2  
**Chave natural:** `cnae_ocupacao`  
**Surrogate key:** `id_ocupacao`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `cnae_ocupacao` | STRING | SIM | Código CNAE da ocupação (chave natural) |
| 2 | `id_ocupacao` | INT | SIM | Surrogate key sequencial |

---

### 10. `dim_modalidade`

**Descrição:** Dimensão de modalidade de crédito do SCR. Combinação de `modalidade` + `submodalidade` como chave natural composta.  
**Registros:** 66  
**Colunas:** 3  
**Chave natural:** `modalidade` + `submodalidade`  
**Surrogate key:** `id_modalidade`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `modalidade` | STRING | SIM | Modalidade de crédito (parte da chave natural) |
| 2 | `submodalidade` | STRING | SIM | Submodalidade (parte da chave natural) |
| 3 | `id_modalidade` | INT | SIM | Surrogate key sequencial |

---

### 11. `dim_conglomerado`

**Descrição:** Dimensão específica do Desenrola. Conglomerados financeiros que participaram das renegociações. Deduplicada por `cod_conglomerado_financeiro`.  
**Registros:** 188  
**Colunas:** 3  
**Chave natural:** `cod_conglomerado_financeiro`  
**Surrogate key:** `id_conglomerado`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `cod_conglomerado_financeiro` | LONG | SIM | Código do conglomerado (chave natural) |
| 2 | `nome_conglomerado_financeiro` | STRING | SIM | Nome do conglomerado financeiro |
| 3 | `id_conglomerado` | INT | SIM | Surrogate key sequencial |

---

### Tabelas Fato

---

### 12. `fct_taxa_de_endividamento`

**Descrição:** Tabela fato de endividamento do SCR. Agrega (`SUM`) todas as medidas financeiras por combinação de 6 chaves dimensionais. Joins com `dim_data_base`, `dim_uf`, `dim_segmento`, `dim_tipo_cliente`, `dim_ocupacao` e `dim_modalidade`. Validação de integridade referencial e duplicação ao final.  
**Registros:** 9.196.768  
**Colunas:** 20  
**Granularidade:** `id_data_base` × `id_uf` × `id_segmento` × `id_tipo_cliente` × `id_ocupacao` × `id_modalidade`  
**Forma de carga:** Joins com dimensões → `groupBy` + `SUM` → `mode=overwrite`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `id_data_base` | INT | SIM | FK → `dim_data_base.id_data_base` |
| 2 | `id_uf` | INT | SIM | FK → `dim_uf.id_uf` |
| 3 | `id_segmento` | INT | SIM | FK → `dim_segmento.id_segmento` |
| 4 | `id_tipo_cliente` | INT | SIM | FK → `dim_tipo_cliente.id_tipo_cliente` |
| 5 | `id_ocupacao` | INT | SIM | FK → `dim_ocupacao.id_ocupacao` |
| 6 | `id_modalidade` | INT | SIM | FK → `dim_modalidade.id_modalidade` |
| 7 | `numero_de_operacoes` | DECIMAL(18,2) | SIM | Soma total de operações |
| 8 | `carteira_ativa` | DECIMAL(18,2) | SIM | Soma da carteira ativa |
| 9 | `carteira_a_vencer` | DECIMAL(18,2) | SIM | Soma da carteira a vencer |
| 10 | `carteira_vencida` | DECIMAL(18,2) | SIM | Soma da carteira vencida |
| 11 | `carteira_inadimplencia` | DECIMAL(18,2) | SIM | Soma da carteira em inadimplência |
| 12 | `ativo_problematico` | DECIMAL(18,2) | SIM | Soma do ativo problemático |
| 13 | `a_vencer_ate_90_dias` | DECIMAL(18,2) | SIM | Soma a vencer até 90 dias |
| 14 | `a_vencer_de_91_ate_360_dias` | DECIMAL(18,2) | SIM | Soma a vencer de 91 a 360 dias |
| 15 | `a_vencer_de_361_ate_1080_dias` | DECIMAL(18,2) | SIM | Soma a vencer de 361 a 1080 dias |
| 16 | `a_vencer_de_1081_ate_1800_dias` | DECIMAL(18,2) | SIM | Soma a vencer de 1081 a 1800 dias |
| 17 | `a_vencer_de_1801_ate_5400_dias` | DECIMAL(18,2) | SIM | Soma a vencer de 1801 a 5400 dias |
| 18 | `a_vencer_acima_de_5400_dias` | DECIMAL(18,2) | SIM | Soma a vencer acima de 5400 dias |
| 19 | `vencido_de_15_ate_90_dias` | DECIMAL(18,2) | SIM | Soma vencido de 15 a 90 dias |
| 20 | `vencido_acima_de_90_dias` | DECIMAL(18,2) | SIM | Soma vencido acima de 90 dias |

---

### 13. `fct_desenrola_negociacao`

**Descrição:** Tabela fato de negociações do Desenrola Brasil. Agrega (`SUM`) operações e volume por 4 chaves dimensionais + `tipo_desenrola`. Converte `VOLUME_OPERACOES` de string (vírgula decimal) para `DECIMAL(18,2)`. Joins com `dim_data_base`, `dim_uf` (compartilhadas) e `dim_conglomerado` (específica).  
**Registros:** 12.751  
**Colunas:** 6  
**Granularidade:** `id_data_base` × `id_uf` × `id_conglomerado` × `tipo_desenrola`  
**Forma de carga:** Joins com dimensões → `groupBy` + `SUM` → `mode=overwrite`

| # | Coluna | Tipo | Nulável | Descrição |
| --- | --- | --- | --- | --- |
| 1 | `id_data_base` | INT | SIM | FK → `dim_data_base.id_data_base` |
| 2 | `id_uf` | INT | SIM | FK → `dim_uf.id_uf` |
| 3 | `id_conglomerado` | INT | SIM | FK → `dim_conglomerado.id_conglomerado` |
| 4 | `tipo_desenrola` | STRING | SIM | Tipo do programa (`PF_ATE_2_SALARIOS`, `PF_ATE_20_MIL`, `PJ_FATURAMENTO_ATE_4_8_MM`) |
| 5 | `numero_operacoes` | LONG | SIM | Soma total de operações de renegociação |
| 6 | `volume_operacoes` | DECIMAL(18,2) | SIM | Soma do volume financeiro das operações |

---

## Resumo Geral

| Camada | Tabela | Registros | Colunas | Papel |
| --- | --- | --- | --- | --- |
| Bronze | `brz_bcb_scr` | 14.194.430 | 29 | Ingestão raw SCR |
| Bronze | `brz_bcb_desenrola` | 12.751 | 11 | Ingestão raw Desenrola |
| Silver | `slv_bcb_scr` | 14.194.430 | 24 | Transformação e tipagem SCR |
| Silver | `slv_bcb_desenrola` | 12.751 | 8 | Transformação e tipagem Desenrola |
| Gold | `dim_data_base` | 84 | 4 | Dimensão temporal (compartilhada) |
| Gold | `dim_uf` | 27 | 3 | Dimensão geográfica (compartilhada) |
| Gold | `dim_segmento` | 8 | 2 | Dimensão de segmento |
| Gold | `dim_tipo_cliente` | 14 | 3 | Dimensão de tipo/porte do cliente |
| Gold | `dim_ocupacao` | 30 | 2 | Dimensão de ocupação (CNAE) |
| Gold | `dim_modalidade` | 66 | 3 | Dimensão de modalidade |
| Gold | `dim_conglomerado` | 188 | 3 | Dimensão de conglomerado financeiro |
| Gold | `fct_taxa_de_endividamento` | 9.196.768 | 20 | Fato de endividamento SCR |
| Gold | `fct_desenrola_negociacao` | 12.751 | 6 | Fato de negociações Desenrola |

# Autoavaliação

O MVP atingiu os objetivos propostos, resultando em um pipeline funcional e organizado nas camadas Bronze, Silver e Gold. Tornei dinamica a busca dos dados para evitar manipulação de arquivos garantindo a consistencia dos dados raw. 
O projeto possibilitou aplicar conceitos de coleta, transformação, modelagem dimensional, qualidade e análise de dados em ambiente de nuvem em uma ferramenta que eu não estava habituado a usar enquanto engenheiro de software.
Os principais desafios envolveram o tratamento do volume de dados do BCB e a garantia de consistência entre as diferentes camadas. Fiz a marcação entre as camadas usando prefixos para as camadas bronze e prata e para a camada ouro usei as dimensões e fatos, entendo que poderia ter segregado por schema ou por catálogo, mas para uma abordagem acadêmica tentei focar mais no processamento da pipeline.
As análises desenvolvidas permitiram responder às perguntas de negócio inicialmente definidas.
Como trabalhos futuros, o projeto pode evoluir com novas fontes de dados, bureaus como serasa, bvs poderiam enriquecer a base para responder perguntas mais ligadas a risco de crédito. Também poderiam ser construídos dashboards para análise dos dados, mas decidi deixar a geração junto com a pipeline de dados por não termos uma ferramenta pública de acesso a esses dashboards.
Achei o trabalho bem produtivo e construiu bastante meu conhecimento.